# ДЕТАЛЬНЫЙ ПЛАН НОУТБУКА: MedVAE 3D Baseline без кода

## ЭТАП 1: РАЗДЕЛЕНИЕ ДАННЫХ

### 1.1 Анализ структуры данных
- Сканирование папки с предобработанными нормальными КТ (425 томов)
- Сканирование папки с тестовыми данными (норма + патологии)
- Подсчет количества файлов в каждой категории
- Проверка соответствия имен файлов и доступности данных

### 1.2 Создание ground truth labels для test set
- Определение логики маркировки файлов (по именам файлов/папкам)
- Создание словаря соответствия filename → label (0=норма, 1=патология)
- Подсчет баланса классов в тестовых данных
- Валидация корректности разметки

### 1.3 Стратегия разделения нормальных данных
- Train split: 80% от нормальных данных (~340 томов)
- Validation split: 20% от нормальных данных (~85 томов) 
- Обеспечение детерминированности разделения (фиксированный seed)
- Сохранение split информации в JSON файл для воспроизводимости

### 1.4 Финальная структура датасетов
- Training dataset: только нормальные КТ для обучения автоэнкодера
- Validation dataset: только нормальные КТ для мониторинга overfitting
- Test dataset: смешанный (норма + патологии) для итогового evaluation
- Создание summary статистики по всем splits

## ЭТАП 2: CPU ТЕСТИРОВАНИЕ

### 2.1 Environment validation
- Проверка доступности всех required библиотек
- Тестирование импорта MedVAE (может потребовать интернет)
- Проверка версий PyTorch, CUDA compatibility
- Валидация рабочего окружения для GPU

### 2.2 Data integrity check
- Загрузка 5-10 случайных файлов из каждого split
- Проверка формата данных (размер, dtype, диапазон значений)
- Валидация что данные соответствуют ожиданиям (128,128,64) и
- Тестирование DataLoader создания на dummy данных

### 2.3 MedVAE model loading test
- Попытка загрузки pretrained MedVAE 3D на CPU
- Тестирование forward pass на dummy tensor (1,1,128,128,64)
- Проверка output размерности и отсутствия NaN/Inf
- Измерение размера модели и количества параметров

### 2.4 Functions dry run
- Создание mock datasets с известными labels
- Тестирование всех utility функций на dummy данных
- Проверка computation reconstruction error без GPU
- Валидация metrics calculation (AUC, F1) на fake scores

### 2.5 Memory estimation
- Расчет ожидаемого GPU memory usage для batch_size 2,3,4
- Проверка что все промежуточные результаты fit в memory
- Estimation времени обучения based на количестве параметров
- Создание fallback плана если memory issues


## ЭТАП 3: DATASET PREPARATION & LOADING

### 3.1 Custom Dataset класс implementation
- Класс для загрузки предобработанных .nii.gz файлов
- Поддержка train/val/test splits через JSON конфигурацию
- Корректное handling размерности тензоров (add channel dimension)
- Error handling для corrupted files

### 3.2 DataLoader configuration
- Train DataLoader: batch_size=3, shuffle=True, оптимизирован для V100
- Validation DataLoader: batch_size=2, shuffle=False, для мониторинга
- Test DataLoader: batch_size=1, shuffle=False, для точного evaluation
- Оптимальные настройки num_workers, pin_memory, prefetch_factor

### 3.3 Data loading validation
- Тестирование загрузки batch из каждого DataLoader
- Проверка корректности размерностей и типов данных
- Валидация что shuffle работает для train set
- Benchmark скорости загрузки данных

## ЭТАП 4: MODEL SETUP & CONFIGURATION

### 4.1 MedVAE model initialization  
- Загрузка pretrained MedVAE 3D с compression_factor=64
- Transfer модели на GPU device
- Анализ архитектуры модели (encoder/decoder structure)
- Проверка что input размерность (128,128,64) supported

### 4.2 Training configuration
- Optimizer setup: AdamW с оптимальными параметрами для fine-tuning
- Loss function: MSE + optional SSIM for better reconstruction quality
- Learning rate scheduler: CosineAnnealingLR для smooth convergence
- Mixed precision setup с GradScaler для V100 optimization

### 4.3 Training monitoring setup
- Loss tracking для train/validation
- Checkpoint saving strategy (каждые 2 эпохи + best model)
- Early stopping criteria (validation loss plateau)
- Intermediate test evaluation для monitoring progress

## ЭТАП 5: TRAINING EXECUTION

### 5.1 Training loop implementation
- Main training loop с progress tracking
- Mixed precision forward/backward pass
- Gradient accumulation если используем
- Memory cleanup для предотвращения OOM

### 5.2 Validation monitoring
- Validation loop после каждой эпохи
- Reconstruction loss tracking на validation set
- Overfitting detection through loss curves
- Best model checkpointing based на validation metrics

### 5.3 Intermediate test evaluation
- Test evaluation каждые 2 эпохи (НЕ влияет на training)
- Anomaly score computation на test set
- AUC calculation для monitoring progress
- Early indication of final performance

## ЭТАП 6: FINAL EVALUATION & RESULTS

### 6.1 Best model loading and inference
- Загрузка best checkpoint based на validation loss
- Full test set evaluation с reconstruction errors
- Anomaly score computation для всех test samples
- Ground truth labels alignment с predictions

### 6.2 Metrics computation
- ROC AUC calculation
- Precision-Recall curve и optimal threshold finding
- Classification report с optimal threshold
- Confusion matrix analysis

### 6.3 Visualizations and analysis
- Training curves (train/val loss, test AUC over epochs)
- ROC curve и PR curve plotting
- Score distributions для normal vs pathology
- Error analysis на misclassified cases

## ЭТАП 7: RESULTS SAVING & EXPORT

### 7.1 Model artifacts saving
- Final trained model weights (best_model.pth)
- Full checkpoint с optimizer state для resuming
- Model configuration и hyperparameters
- Training history logs

### 7.2 Results export
- Detailed results JSON с all metrics и scores
- Per-sample predictions с filenames
- Summary statistics и performance breakdown
- Visualization plots saving

### 7.3 Reproducibility package
- Complete configuration file для reproducing results
- Dataset split information
- Random seeds и environment details
- Instructions для model loading и inference



# Environment Setup

In [ ]:
# %pip install medvae
# %pip install scikit-learn matplotlib seaborn
# %pip install nibabel 

Unknown instance spec: Please select VM configuration

In [1]:
# ===================================
# УСТАНОВКА И ИМПОРТЫ
# ===================================

# Стандартная библиотека Python
import json
import os
import random
import time
import warnings
from collections import defaultdict
from functools import partial
from pathlib import Path

# Сторонние библиотеки
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
import torch.multiprocessing as mp
import torch.optim as optim
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    roc_curve
)
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from torch.cuda.amp import GradScaler, autocast
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

# Настройки
warnings.filterwarnings('ignore')

In [2]:
# установим рабочую директорию
work_dir = os.path.join(os.path.expanduser('~'), 'work', 'data')

%cd {work_dir}

/home/jupyter/work/data


In [3]:
# ЯЧЕЙКА: ИСПРАВЛЕННЫЙ DEVICE HANDLING
print("🔧 FIXING DEVICE MISMATCH ERRORS")
print("="*60)

import torch
import torch.nn as nn

# Ensure consistent device handling
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {device}")

# Fix MedVAE model loading with proper device handling
try:
    from medvae.models import AutoencoderKL_3D
    
    # Configuration
    ddconfig = {
        "double_z": True,
        "z_channels": 4,
        "resolution": 128,
        "in_channels": 1,
        "out_ch": 1,
        "ch": 128,
        "ch_mult": [1, 2, 4, 4],
        "num_res_blocks": 2,
        "attn_resolutions": [],
        "dropout": 0.0,
    }
    
    embed_dim = 4
    
    print("🔄 Loading MedVAE with fixed device handling...")
    medvae_model = AutoencoderKL_3D(ddconfig=ddconfig, embed_dim=embed_dim)
    
    # CRITICAL FIX: Move to device BEFORE any operations
    medvae_model = medvae_model.to(device)
    medvae_model.eval()
    
    print("✅ MedVAE loaded successfully!")
    
    # Wrapper class with FIXED device handling
    class MedVAEWrapper(nn.Module):
        def __init__(self, base_model):
            super().__init__()
            self.base_model = base_model
            # Ensure wrapper is on same device
            self.to(next(base_model.parameters()).device)
        
        def forward(self, x):
            # Ensure input is on correct device
            if x.device != next(self.parameters()).device:
                x = x.to(next(self.parameters()).device)
            
            # VAE returns tuple (reconstruction, posterior_info)
            output = self.base_model(x)
            
            if isinstance(output, tuple):
                reconstruction = output[0]
                return reconstruction
            else:
                return output
        
        def encode(self, x):
            if x.device != next(self.parameters()).device:
                x = x.to(next(self.parameters()).device)
            return self.base_model.encode(x)
        
        def decode(self, z):
            if z.device != next(self.parameters()).device:
                z = z.to(next(self.parameters()).device)
            return self.base_model.decode(z)
    
    # Create wrapper
    medvae_model = MedVAEWrapper(medvae_model)
    
    print("✅ Device-safe wrapper created!")
    medvae_available = True
    
except Exception as e:
    print(f"❌ MedVAE loading failed: {e}")
    medvae_available = False


🔧 FIXING DEVICE MISMATCH ERRORS
✅ Device: cuda


2025-09-30 22:02:39.009739: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-30 22:02:43.811769: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


🔄 Loading MedVAE with fixed device handling...
making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 4, 16, 16) = 1024 dimensions.
making attention of type 'vanilla' with 512 in_channels
✅ MedVAE loaded successfully!
✅ Device-safe wrapper created!


# ЭТАП 1: РАЗДЕЛЕНИЕ ДАННЫХ

## 1.1 Анализ структуры данных

- Сканирование папки с предобработанными нормальными КТ (425 томов)

- Сканирование папки с тестовыми данными (норма + патологии)

- Подсчет количества файлов в каждой категории

- Проверка соответствия имен файлов и доступности данных

In [4]:
print("📊 АНАЛИЗ СТРУКТУРЫ ПРЕДОБРАБОТАННЫХ ДАННЫХ")
print("=" * 60)

# Пути к вашим данным
NORMAL_DATA_PATH = Path("preprocessed_data/normal/612a52a0b6a8")
PATHOLOGY_DATA_PATH = Path("preprocessed_data/pathology/612a52a0b6a8") 

# Проверяем существование папок
print("🔍 ПРОВЕРКА ПУТЕЙ:")
if NORMAL_DATA_PATH.exists():
    print(f"   ✅ Нормы найдены: {NORMAL_DATA_PATH}")
else:
    print(f"   ❌ Нормы НЕ найдены: {NORMAL_DATA_PATH}")

if PATHOLOGY_DATA_PATH.exists():
    print(f"   ✅ Патологии найдены: {PATHOLOGY_DATA_PATH}")
else:
    print(f"   ❌ Патологии НЕ найдены: {PATHOLOGY_DATA_PATH}")

# Сканируем нормальные данные
print(f"\n📁 АНАЛИЗ НОРМАЛЬНЫХ ДАННЫХ:")
normal_files = list(NORMAL_DATA_PATH.glob("*.nii.gz"))
print(f"   Количество файлов: {len(normal_files)}")

if len(normal_files) > 0:
    # Показываем первые 5 файлов
    print("   Примеры файлов:")
    for i, file_path in enumerate(normal_files[:5]):
        file_size_mb = file_path.stat().st_size / (1024 * 1024)
        print(f"      [{i+1}] {file_path.name} ({file_size_mb:.1f} MB)")
    
    if len(normal_files) > 5:
        print(f"      ... и ещё {len(normal_files) - 5} файлов")

# Сканируем патологические данные
print(f"\n📁 АНАЛИЗ ПАТОЛОГИЧЕСКИХ ДАННЫХ:")
pathology_files = list(PATHOLOGY_DATA_PATH.glob("*.nii.gz"))
print(f"   Количество файлов: {len(pathology_files)}")

if len(pathology_files) > 0:
    # Показываем первые 5 файлов
    print("   Примеры файлов:")
    for i, file_path in enumerate(pathology_files[:5]):
        file_size_mb = file_path.stat().st_size / (1024 * 1024)
        print(f"      [{i+1}] {file_path.name} ({file_size_mb:.1f} MB)")
    
    if len(pathology_files) > 5:
        print(f"      ... и ещё {len(pathology_files) - 5} файлов")

# Общая статистика
print(f"\n📊 ОБЩАЯ СТАТИСТИКА:")
total_files = len(normal_files) + len(pathology_files)
print(f"   Всего файлов: {total_files}")
print(f"   Нормальных: {len(normal_files)} ({len(normal_files)/total_files*100:.1f}%)")
print(f"   Патологических: {len(pathology_files)} ({len(pathology_files)/total_files*100:.1f}%)")

# Анализ размеров файлов
if normal_files:
    normal_sizes = [f.stat().st_size / (1024 * 1024) for f in normal_files[:10]]  # первые 10
    print(f"   Размер нормальных файлов (MB): {np.mean(normal_sizes):.1f} ± {np.std(normal_sizes):.1f}")

if pathology_files:
    path_sizes = [f.stat().st_size / (1024 * 1024) for f in pathology_files[:10]]  # первые 10
    print(f"   Размер патологических файлов (MB): {np.mean(path_sizes):.1f} ± {np.std(path_sizes):.1f}")

print(f"\n✅ СТРУКТУРА ДАННЫХ ПРОАНАЛИЗИРОВАНА")


📊 АНАЛИЗ СТРУКТУРЫ ПРЕДОБРАБОТАННЫХ ДАННЫХ
🔍 ПРОВЕРКА ПУТЕЙ:
   ✅ Нормы найдены: preprocessed_data/normal/612a52a0b6a8
   ✅ Патологии найдены: preprocessed_data/pathology/612a52a0b6a8

📁 АНАЛИЗ НОРМАЛЬНЫХ ДАННЫХ:
   Количество файлов: 419
   Примеры файлов:
      [1] 7ab789ef.nii.gz (2.5 MB)
      [2] 7ad0f919.nii.gz (2.5 MB)
      [3] 759e9bab.nii.gz (2.5 MB)
      [4] 10e0a4d2.nii.gz (2.9 MB)
      [5] 9ebed2e0.nii.gz (2.5 MB)
      ... и ещё 414 файлов

📁 АНАЛИЗ ПАТОЛОГИЧЕСКИХ ДАННЫХ:
   Количество файлов: 372
   Примеры файлов:
      [1] dbec1ebe.nii.gz (2.4 MB)
      [2] c8124ccd.nii.gz (2.6 MB)
      [3] 670a7641.nii.gz (2.5 MB)
      [4] 85272cc9.nii.gz (2.6 MB)
      [5] 2af91d07.nii.gz (2.6 MB)
      ... и ещё 367 файлов

📊 ОБЩАЯ СТАТИСТИКА:
   Всего файлов: 791
   Нормальных: 419 (53.0%)
   Патологических: 372 (47.0%)
   Размер нормальных файлов (MB): 2.5 ± 0.1
   Размер патологических файлов (MB): 2.6 ± 0.1

✅ СТРУКТУРА ДАННЫХ ПРОАНАЛИЗИРОВАНА


## 1.2: Создание ground truth labels для test set
Что мы делаем:
Создаем словарь соответствия filename → label для всех файлов. Поскольку у нас четкое разделение по папкам, это будет просто.

Детали реализации:
Логика маркировки:
- Все файлы из папки normal/ → label = 0

- Все файлы из папки pathology/ → label = 1

Что код будет делать:
- Создает единый словарь {filename: label} для всех 791 файлов

- Проверяет уникальность имен файлов (нет дублирования между папками)

- Создает ground truth mapping для использования в дальнейшем

- Выводит статистику по классам



In [5]:
# ===================================
# СОЗДАНИЕ GROUND TRUTH LABELS
# ===================================

print("🏷️  СОЗДАНИЕ GROUND TRUTH LABELS")
print("=" * 60)

# Создаем mapping filename -> label
ground_truth_labels = {}

# Добавляем нормальные файлы (label = 0)
print("🟢 Обрабатываем нормальные файлы...")
for file_path in normal_files:
    filename = file_path.name
    ground_truth_labels[filename] = 0

print(f"   Добавлено нормальных: {len(normal_files)}")

# Добавляем патологические файлы (label = 1)  
print("🔴 Обрабатываем патологические файлы...")
for file_path in pathology_files:
    filename = file_path.name
    
    # Проверяем на дублирование имен
    if filename in ground_truth_labels:
        print(f"   ⚠️ ДУБЛИРОВАНИЕ: {filename} найден в обеих папках!")
    else:
        ground_truth_labels[filename] = 1

print(f"   Добавлено патологических: {len(pathology_files)}")

# Проверяем целостность
print(f"\n📊 GROUND TRUTH СТАТИСТИКА:")
total_labels = len(ground_truth_labels)
normal_count = sum(1 for label in ground_truth_labels.values() if label == 0)
pathology_count = sum(1 for label in ground_truth_labels.values() if label == 1)

print(f"   Всего файлов в ground truth: {total_labels}")
print(f"   Нормальных (label=0): {normal_count} ({normal_count/total_labels*100:.1f}%)")
print(f"   Патологических (label=1): {pathology_count} ({pathology_count/total_labels*100:.1f}%)")

# Проверяем ожидаемые числа
expected_total = len(normal_files) + len(pathology_files)
if total_labels == expected_total:
    print(f"   ✅ Количество labels совпадает с количеством файлов")
else:
    print(f"   ❌ ОШИБКА: Ожидали {expected_total}, получили {total_labels}")

# Показываем примеры
print(f"\n📝 ПРИМЕРЫ GROUND TRUTH MAPPING:")
sample_items = list(ground_truth_labels.items())[:5]
for filename, label in sample_items:
    label_text = "NORMAL" if label == 0 else "PATHOLOGY"
    print(f"   {filename} → {label} ({label_text})")

print(f"\n✅ GROUND TRUTH LABELS СОЗДАНЫ")

# Сохраняем в переменной для дальнейшего использования
print(f"💾 Ground truth сохранен в переменной 'ground_truth_labels'")

🏷️  СОЗДАНИЕ GROUND TRUTH LABELS
🟢 Обрабатываем нормальные файлы...
   Добавлено нормальных: 419
🔴 Обрабатываем патологические файлы...
   Добавлено патологических: 372

📊 GROUND TRUTH СТАТИСТИКА:
   Всего файлов в ground truth: 791
   Нормальных (label=0): 419 (53.0%)
   Патологических (label=1): 372 (47.0%)
   ✅ Количество labels совпадает с количеством файлов

📝 ПРИМЕРЫ GROUND TRUTH MAPPING:
   7ab789ef.nii.gz → 0 (NORMAL)
   7ad0f919.nii.gz → 0 (NORMAL)
   759e9bab.nii.gz → 0 (NORMAL)
   10e0a4d2.nii.gz → 0 (NORMAL)
   9ebed2e0.nii.gz → 0 (NORMAL)

✅ GROUND TRUTH LABELS СОЗДАНЫ
💾 Ground truth сохранен в переменной 'ground_truth_labels'


## 1.3: Стратегия разделения нормальных данных
Что мы делаем:
Делим 419 нормальных файлов на train (80%) и validation (20%) splits для обучения автоэнкодера. Тестовый set будет включать ВСЕ файлы (нормы + патологии).

Детали реализации:
Стратегия разделения:
- Training set (только нормы - максимизируем):
75% от нормальных = 419 × 0.75 = 314 нормальных томов

Максимум данных для изучения нормальной анатомии

- Validation set (для threshold tuning + early stopping):
10% от нормальных = 419 × 0.10 = 42 нормальных тома

50% от патологий = 372 × 0.50 = 186 патологических томов

Итого val: 228 томов (18% норма, 82% патология)

- Test set (финальная честная оценка):
15% от нормальных = 419 × 0.15 = 63 нормальных тома

50% от патологий = 372 × 0.50 = 186 патологических томов

Итого test: 249 томов (25% норма, 75% патология)

Важные детали:
Используем фиксированный random seed для воспроизводимости

Сохраняем split информацию в JSON для будущего использования

Проверяем что нет пересечений между train/val

In [6]:
print("📊 СТРАТЕГИЯ РАЗДЕЛЕНИЯ ДАННЫХ ДЛЯ ANOMALY DETECTION")
print("=" * 60)

# Фиксируем random seed для воспроизводимости
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"🎲 Random seed: {RANDOM_SEED}")

# Извлекаем списки файлов для разделения
normal_filenames = [f.name for f in normal_files]
pathology_filenames = [f.name for f in pathology_files]

print(f"\n📁 ИСХОДНЫЕ ДАННЫЕ:")
print(f"   Нормальных файлов: {len(normal_filenames)}")
print(f"   Патологических файлов: {len(pathology_filenames)}")

# РАЗДЕЛЕНИЕ НОРМАЛЬНЫХ ДАННЫХ
# Train: 75%, Validation: 10%, Test: 15%
print(f"\n🔄 РАЗДЕЛЕНИЕ НОРМАЛЬНЫХ ДАННЫХ:")

# Сначала отделяем test (15%)
normal_train_val, normal_test = train_test_split(
    normal_filenames, 
    test_size=0.15,  # 15% на test
    random_state=RANDOM_SEED
)

# Теперь делим train_val на train (75%) и validation (10%)
# Из оставшихся 85%: train = 75/85 = 88.2%, val = 10/85 = 11.8%
normal_train, normal_val = train_test_split(
    normal_train_val,
    test_size=0.118,  # 10% от общих (11.8% от train_val)
    random_state=RANDOM_SEED
)

print(f"   Train: {len(normal_train)} томов ({len(normal_train)/len(normal_filenames)*100:.1f}%)")
print(f"   Validation: {len(normal_val)} томов ({len(normal_val)/len(normal_filenames)*100:.1f}%)")
print(f"   Test: {len(normal_test)} томов ({len(normal_test)/len(normal_filenames)*100:.1f}%)")

# РАЗДЕЛЕНИЕ ПАТОЛОГИЧЕСКИХ ДАННЫХ  
# Validation: 50%, Test: 50%
print(f"\n🔄 РАЗДЕЛЕНИЕ ПАТОЛОГИЧЕСКИХ ДАННЫХ:")

pathology_val, pathology_test = train_test_split(
    pathology_filenames,
    test_size=0.5,  # 50% на test, 50% на validation
    random_state=RANDOM_SEED
)

print(f"   Validation: {len(pathology_val)} томов ({len(pathology_val)/len(pathology_filenames)*100:.1f}%)")
print(f"   Test: {len(pathology_test)} томов ({len(pathology_test)/len(pathology_filenames)*100:.1f}%)")

# ИТОГОВЫЕ НАБОРЫ ДАННЫХ
print(f"\n📊 ФИНАЛЬНЫЕ НАБОРЫ:")

# Training set (только нормы)
train_files = normal_train
print(f"   🟢 TRAINING: {len(train_files)} томов (100% норма)")

# Validation set (нормы + патологии)  
val_files = normal_val + pathology_val
val_normal_count = len(normal_val)
val_pathology_count = len(pathology_val)
print(f"   🔶 VALIDATION: {len(val_files)} томов ({val_normal_count} норма, {val_pathology_count} патология)")
print(f"       Баланс: {val_normal_count/len(val_files)*100:.1f}% норма, {val_pathology_count/len(val_files)*100:.1f}% патология")

# Test set (нормы + патологии)
test_files = normal_test + pathology_test  
test_normal_count = len(normal_test)
test_pathology_count = len(pathology_test)
print(f"   🔴 TEST: {len(test_files)} томов ({test_normal_count} норма, {test_pathology_count} патология)")
print(f"       Баланс: {test_normal_count/len(test_files)*100:.1f}% норма, {test_pathology_count/len(test_files)*100:.1f}% патология")

# Проверка общего количества
total_files = len(train_files) + len(val_files) + len(test_files)
expected_total = len(normal_filenames) + len(pathology_filenames)

print(f"\n✅ ПРОВЕРКА ЦЕЛОСТНОСТИ:")
print(f"   Всего файлов в splits: {total_files}")
print(f"   Ожидаемое общее: {expected_total}")
print(f"   Проверка: {'✅ ОК' if total_files == expected_total else '❌ ОШИБКА'}")

print(f"\n💾 Splits сохранены в переменных: train_files, val_files, test_files")


📊 СТРАТЕГИЯ РАЗДЕЛЕНИЯ ДАННЫХ ДЛЯ ANOMALY DETECTION
🎲 Random seed: 42

📁 ИСХОДНЫЕ ДАННЫЕ:
   Нормальных файлов: 419
   Патологических файлов: 372

🔄 РАЗДЕЛЕНИЕ НОРМАЛЬНЫХ ДАННЫХ:
   Train: 313 томов (74.7%)
   Validation: 43 томов (10.3%)
   Test: 63 томов (15.0%)

🔄 РАЗДЕЛЕНИЕ ПАТОЛОГИЧЕСКИХ ДАННЫХ:
   Validation: 186 томов (50.0%)
   Test: 186 томов (50.0%)

📊 ФИНАЛЬНЫЕ НАБОРЫ:
   🟢 TRAINING: 313 томов (100% норма)
   🔶 VALIDATION: 229 томов (43 норма, 186 патология)
       Баланс: 18.8% норма, 81.2% патология
   🔴 TEST: 249 томов (63 норма, 186 патология)
       Баланс: 25.3% норма, 74.7% патология

✅ ПРОВЕРКА ЦЕЛОСТНОСТИ:
   Всего файлов в splits: 791
   Ожидаемое общее: 791
   Проверка: ✅ ОК

💾 Splits сохранены в переменных: train_files, val_files, test_files


## 1.4: Финальная структура датасетов
Что мы делаем:
Создаем JSON файл с информацией о splits для воспроизводимости и создаем summary статистику по всем наборам данных.

In [7]:
print("📋 СОЗДАНИЕ ФИНАЛЬНОЙ СТРУКТУРЫ ДАТАСЕТОВ")
print("=" * 60)

# Создаем структурированную информацию о splits
dataset_splits = {
    "dataset_info": {
        "total_normal_files": len(normal_filenames),
        "total_pathology_files": len(pathology_filenames),
        "total_files": len(normal_filenames) + len(pathology_filenames),
        "normal_data_path": str(NORMAL_DATA_PATH),
        "pathology_data_path": str(PATHOLOGY_DATA_PATH),
        "created_date": "2025-09-28",
        "random_seed": RANDOM_SEED
    },
    
    "train": {
        "files": train_files,
        "count": len(train_files),
        "normal_count": len(train_files),
        "pathology_count": 0,
        "description": "Training set - только нормальные КТ для обучения автоэнкодера"
    },
    
    "validation": {
        "files": val_files,
        "count": len(val_files),
        "normal_count": len(normal_val),
        "pathology_count": len(pathology_val),
        "normal_files": normal_val,
        "pathology_files": pathology_val,
        "description": "Validation set - для early stopping (нормы) и threshold optimization (патологии)"
    },
    
    "test": {
        "files": test_files,
        "count": len(test_files),  
        "normal_count": len(normal_test),
        "pathology_count": len(pathology_test),
        "normal_files": normal_test,
        "pathology_files": pathology_test,
        "description": "Test set - финальная оценка на holdout данных"
    }
}

# Детальная статистика
print("📊 ДЕТАЛЬНАЯ СТАТИСТИКА ПО SPLITS:")

for split_name, split_info in dataset_splits.items():
    if split_name == "dataset_info":
        continue
        
    normal_pct = (split_info["normal_count"] / split_info["count"]) * 100
    pathology_pct = (split_info["pathology_count"] / split_info["count"]) * 100
    
    print(f"\n🔹 {split_name.upper()}:")
    print(f"   Всего: {split_info['count']} томов")
    print(f"   Нормальные: {split_info['normal_count']} ({normal_pct:.1f}%)")
    print(f"   Патологические: {split_info['pathology_count']} ({pathology_pct:.1f}%)")
    print(f"   Описание: {split_info['description']}")

# Создаем mapping filename -> split для quick lookup
file_to_split = {}

for filename in train_files:
    file_to_split[filename] = "train"
    
for filename in val_files:
    if filename in normal_val:
        file_to_split[filename] = "validation_normal"
    else:
        file_to_split[filename] = "validation_pathology"
        
for filename in test_files:
    if filename in normal_test:
        file_to_split[filename] = "test_normal"  
    else:
        file_to_split[filename] = "test_pathology"

# Добавляем mapping в структуру
dataset_splits["file_to_split_mapping"] = file_to_split

# Сохраняем splits в JSON файл для воспроизводимости
splits_filename = "dataset_splits.json"

print(f"\n💾 СОХРАНЕНИЕ SPLITS:")
try:
    with open(splits_filename, 'w', encoding='utf-8') as f:
        json.dump(dataset_splits, f, indent=2, ensure_ascii=False)
    print(f"   ✅ Splits сохранены в файл: {splits_filename}")
    print(f"   📁 Размер файла: {os.path.getsize(splits_filename) / 1024:.1f} KB")
except Exception as e:
    print(f"   ❌ Ошибка сохранения: {e}")

# Проверяем пересечения между splits (не должно быть!)
print(f"\n🔍 ПРОВЕРКА ПЕРЕСЕЧЕНИЙ:")

train_set = set(train_files)
val_set = set(val_files)
test_set = set(test_files)

train_val_overlap = train_set.intersection(val_set)
train_test_overlap = train_set.intersection(test_set)
val_test_overlap = val_set.intersection(test_set)

print(f"   Train ∩ Validation: {len(train_val_overlap)} файлов {'✅' if len(train_val_overlap) == 0 else '❌'}")
print(f"   Train ∩ Test: {len(train_test_overlap)} файлов {'✅' if len(train_test_overlap) == 0 else '❌'}")
print(f"   Validation ∩ Test: {len(val_test_overlap)} файлов {'✅' if len(val_test_overlap) == 0 else '❌'}")

if len(train_val_overlap) > 0 or len(train_test_overlap) > 0 or len(val_test_overlap) > 0:
    print("   ⚠️ ОБНАРУЖЕНЫ ПЕРЕСЕЧЕНИЯ! Проверьте логику разделения.")
else:
    print("   ✅ Пересечений не обнаружено - splits корректные")

print(f"\n🎯 ГОТОВНОСТЬ К ОБУЧЕНИЮ:")
print(f"   📊 Dataset splits созданы и сохранены")
print(f"   📋 Ground truth labels готовы")  
print(f"   🔄 Splits без пересечений")
print(f"   ✅ ЭТАП 1 ЗАВЕРШЕН - переходим к CPU тестированию")

# Сохраняем ключевые переменные для следующих этапов
print(f"\n💾 Переменные готовы для использования:")
print(f"   - ground_truth_labels: {len(ground_truth_labels)} entries")
print(f"   - dataset_splits: полная информация о разделении")
print(f"   - train_files, val_files, test_files: списки файлов для каждого split")


📋 СОЗДАНИЕ ФИНАЛЬНОЙ СТРУКТУРЫ ДАТАСЕТОВ
📊 ДЕТАЛЬНАЯ СТАТИСТИКА ПО SPLITS:

🔹 TRAIN:
   Всего: 313 томов
   Нормальные: 313 (100.0%)
   Патологические: 0 (0.0%)
   Описание: Training set - только нормальные КТ для обучения автоэнкодера

🔹 VALIDATION:
   Всего: 229 томов
   Нормальные: 43 (18.8%)
   Патологические: 186 (81.2%)
   Описание: Validation set - для early stopping (нормы) и threshold optimization (патологии)

🔹 TEST:
   Всего: 249 томов
   Нормальные: 63 (25.3%)
   Патологические: 186 (74.7%)
   Описание: Test set - финальная оценка на holdout данных

💾 СОХРАНЕНИЕ SPLITS:
   ✅ Splits сохранены в файл: dataset_splits.json
   📁 Размер файла: 62.0 KB

🔍 ПРОВЕРКА ПЕРЕСЕЧЕНИЙ:
   Train ∩ Validation: 0 файлов ✅
   Train ∩ Test: 0 файлов ✅
   Validation ∩ Test: 0 файлов ✅
   ✅ Пересечений не обнаружено - splits корректные

🎯 ГОТОВНОСТЬ К ОБУЧЕНИЮ:
   📊 Dataset splits созданы и сохранены
   📋 Ground truth labels готовы
   🔄 Splits без пересечений
   ✅ ЭТАП 1 ЗАВЕРШЕН - переходим к CP

# ЭТАП 2: CPU ТЕСТИРОВАНИЕ

## 2.1: Environment validation
Что мы делаем:
Проверяем все библиотеки и зависимости на CPU, чтобы потом не тратить драгоценное время GPU на debugging.

In [8]:
# ===================================
# ЯЧЕЙКА 2.1: ENVIRONMENT VALIDATION  
# ===================================

print("🧪 ПРОВЕРКА РАБОЧЕГО ОКРУЖЕНИЯ")
print("=" * 60)

# Проверяем основные библиотеки
print("📦 ПРОВЕРКА ОСНОВНЫХ БИБЛИОТЕК:")

try:
    import torch
    print(f"   ✅ PyTorch: {torch.__version__}")
    
    # Проверяем CUDA availability (даже на CPU машине)
    cuda_available = torch.cuda.is_available()
    print(f"   🔥 CUDA available: {'✅ Да' if cuda_available else '❌ Нет'}")
    
    if cuda_available:
        print(f"       CUDA version: {torch.version.cuda}")
        print(f"       GPU count: {torch.cuda.device_count()}")
    else:
        print(f"       ℹ️ Работаем на CPU (нормально для тестирования)")
        
except ImportError as e:
    print(f"   ❌ PyTorch: Ошибка импорта - {e}")

try:
    import numpy as np
    print(f"   ✅ NumPy: {np.__version__}")
except ImportError as e:
    print(f"   ❌ NumPy: {e}")

try:
    import nibabel as nib
    print(f"   ✅ NiBabel: {nib.__version__}")
except ImportError as e:
    print(f"   ❌ NiBabel: {e} - установите: pip install nibabel")

try:
    from sklearn.metrics import roc_auc_score, roc_curve
    print(f"   ✅ Scikit-learn: импорт успешен")
except ImportError as e:
    print(f"   ❌ Scikit-learn: {e}")

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    print(f"   ✅ Matplotlib, Seaborn: импорт успешен")
except ImportError as e:
    print(f"   ❌ Visualization libs: {e}")


# Проверяем дополнительные библиотеки для training
print(f"\n🏋️ ПРОВЕРКА TRAINING БИБЛИОТЕК:")

try:
    from torch.utils.data import Dataset, DataLoader
    print("   ✅ PyTorch DataLoader: OK")
except ImportError as e:
    print(f"   ❌ PyTorch DataLoader: {e}")

try:  
    from torch.cuda.amp import autocast, GradScaler
    print("   ✅ Mixed Precision (AMP): OK")
except ImportError as e:
    print(f"   ❌ Mixed Precision: {e}")

try:
    from tqdm import tqdm
    print("   ✅ Progress bars (tqdm): OK")
except ImportError as e:
    print(f"   ❌ tqdm: {e} - установите: pip install tqdm")

# Системная информация
print(f"\n💻 СИСТЕМНАЯ ИНФОРМАЦИЯ:")
print(f"   Python: {__import__('sys').version.split()[0]}")

import platform
print(f"   ОС: {platform.system()} {platform.release()}")

# Проверяем доступную память
import psutil
memory = psutil.virtual_memory()
print(f"   RAM: {memory.total / (1024**3):.1f} GB (доступно: {memory.available / (1024**3):.1f} GB)")

# CPU информация
print(f"   CPU cores: {psutil.cpu_count()}")

print(f"\n🎯 ГОТОВНОСТЬ ОКРУЖЕНИЯ:")

# Критические проверки
critical_checks = [
    ("PyTorch", 'torch' in globals()),
    ("NumPy", 'np' in globals()),
    ("NiBabel", 'nib' in globals()),
    ("Sklearn", 'roc_auc_score' in globals())
]

all_critical_ok = True
for check_name, check_result in critical_checks:
    status = "✅" if check_result else "❌"
    print(f"   {status} {check_name}")
    if not check_result:
        all_critical_ok = False

print(f"\n{'✅ ОКРУЖЕНИЕ ГОТОВО К GPU ЗАПУСКУ' if all_critical_ok else '❌ ИСПРАВЬТЕ КРИТИЧЕСКИЕ ОШИБКИ'}")

if all_critical_ok:
    print("   🚀 Можно переходить к следующему этапу")
else:
    print("   ⚠️ Установите недостающие библиотеки перед продолжением")


🧪 ПРОВЕРКА РАБОЧЕГО ОКРУЖЕНИЯ
📦 ПРОВЕРКА ОСНОВНЫХ БИБЛИОТЕК:
   ✅ PyTorch: 2.8.0+cu128
   🔥 CUDA available: ✅ Да
       CUDA version: 12.8
       GPU count: 1
   ✅ NumPy: 1.26.4
   ✅ NiBabel: 5.3.2
   ✅ Scikit-learn: импорт успешен
   ✅ Matplotlib, Seaborn: импорт успешен

🏋️ ПРОВЕРКА TRAINING БИБЛИОТЕК:
   ✅ PyTorch DataLoader: OK
   ✅ Mixed Precision (AMP): OK
   ✅ Progress bars (tqdm): OK

💻 СИСТЕМНАЯ ИНФОРМАЦИЯ:
   Python: 3.10.12
   ОС: Linux 5.13.0-40-generic
   RAM: 94.3 GB (доступно: 91.4 GB)
   CPU cores: 8

🎯 ГОТОВНОСТЬ ОКРУЖЕНИЯ:
   ✅ PyTorch
   ✅ NumPy
   ✅ NiBabel
   ✅ Sklearn

✅ ОКРУЖЕНИЕ ГОТОВО К GPU ЗАПУСКУ
   🚀 Можно переходить к следующему этапу


## 2.2: Data integrity check
Что мы делаем:
Проверяем целостность наших предобработанных данных - загружаем несколько случайных файлов и проверяем их формат, размеры и значения.

In [9]:
# ===================================
# ЯЧЕЙКА 2.2: DATA INTEGRITY CHECK
# ===================================

print("🔍 ПРОВЕРКА ЦЕЛОСТНОСТИ ДАННЫХ")
print("=" * 60)

import random
import nibabel as nib

# Функция для проверки одного файла
def check_single_file(file_path, expected_shape=(128, 128, 64)):
    """Проверка одного .nii.gz файла"""
    try:
        # Загружаем файл
        nii_img = nib.load(str(file_path))
        volume = nii_img.get_fdata()
        
        # Проверяем основные характеристики
        file_info = {
            "filename": file_path.name,
            "shape": volume.shape,
            "dtype": volume.dtype,
            "min_value": float(volume.min()),
            "max_value": float(volume.max()),
            "mean_value": float(volume.mean()),
            "std_value": float(volume.std()),
            "file_size_mb": file_path.stat().st_size / (1024 * 1024),
            "expected_shape_ok": volume.shape == expected_shape,
            "value_range_ok": 0 <= volume.min() and volume.max() <= 1.0,
            "no_nan": not np.isnan(volume).any(),
            "no_inf": not np.isinf(volume).any()
        }
        
        return file_info, None
        
    except Exception as e:
        return None, str(e)

# Проверяем нормальные файлы  
print("🟢 ПРОВЕРКА НОРМАЛЬНЫХ ФАЙЛОВ:")
sample_normal_files = random.sample(normal_files, min(5, len(normal_files)))

normal_check_results = []
for i, file_path in enumerate(sample_normal_files):
    print(f"   [{i+1}/5] Проверяем: {file_path.name}")
    
    file_info, error = check_single_file(file_path)
    if error:
        print(f"       ❌ Ошибка: {error}")
        continue
    
    normal_check_results.append(file_info)
    
    # Выводим информацию о файле
    shape_status = "✅" if file_info["expected_shape_ok"] else "❌"  
    range_status = "✅" if file_info["value_range_ok"] else "❌"
    nan_status = "✅" if file_info["no_nan"] else "❌"
    
    print(f"       Shape: {file_info['shape']} {shape_status}")
    print(f"       Range: [{file_info['min_value']:.3f}, {file_info['max_value']:.3f}] {range_status}")
    print(f"       Mean±Std: {file_info['mean_value']:.3f}±{file_info['std_value']:.3f}")
    print(f"       Size: {file_info['file_size_mb']:.1f} MB")
    print(f"       No NaN/Inf: {nan_status}")

# Проверяем патологические файлы
print(f"\n🔴 ПРОВЕРКА ПАТОЛОГИЧЕСКИХ ФАЙЛОВ:")
sample_pathology_files = random.sample(pathology_files, min(5, len(pathology_files)))

pathology_check_results = []
for i, file_path in enumerate(sample_pathology_files):
    print(f"   [{i+1}/5] Проверяем: {file_path.name}")
    
    file_info, error = check_single_file(file_path)
    if error:
        print(f"       ❌ Ошибка: {error}")
        continue
        
    pathology_check_results.append(file_info)
    
    # Выводим информацию о файле
    shape_status = "✅" if file_info["expected_shape_ok"] else "❌"  
    range_status = "✅" if file_info["value_range_ok"] else "❌"
    nan_status = "✅" if file_info["no_nan"] else "❌"
    
    print(f"       Shape: {file_info['shape']} {shape_status}")
    print(f"       Range: [{file_info['min_value']:.3f}, {file_info['max_value']:.3f}] {range_status}")
    print(f"       Mean±Std: {file_info['mean_value']:.3f}±{file_info['std_value']:.3f}")
    print(f"       Size: {file_info['file_size_mb']:.1f} MB")
    print(f"       No NaN/Inf: {nan_status}")

# Агрегированная статистика
print(f"\n📊 АГРЕГИРОВАННАЯ СТАТИСТИКА:")

all_results = normal_check_results + pathology_check_results
if len(all_results) > 0:
    
    # Проверки
    shape_ok_count = sum(1 for r in all_results if r["expected_shape_ok"])
    range_ok_count = sum(1 for r in all_results if r["value_range_ok"])  
    no_nan_count = sum(1 for r in all_results if r["no_nan"])
    no_inf_count = sum(1 for r in all_results if r["no_inf"])
    
    print(f"   Проверено файлов: {len(all_results)}")
    print(f"   Правильная размерность: {shape_ok_count}/{len(all_results)} ({shape_ok_count/len(all_results)*100:.1f}%)")
    print(f"   Правильный диапазон [0,1]: {range_ok_count}/{len(all_results)} ({range_ok_count/len(all_results)*100:.1f}%)")
    print(f"   Без NaN: {no_nan_count}/{len(all_results)} ({no_nan_count/len(all_results)*100:.1f}%)")
    print(f"   Без Inf: {no_inf_count}/{len(all_results)} ({no_inf_count/len(all_results)*100:.1f}%)")
    
    # Статистика размеров
    file_sizes = [r["file_size_mb"] for r in all_results]
    print(f"   Размер файлов: {np.mean(file_sizes):.1f}±{np.std(file_sizes):.1f} MB")
    
    # Статистика значений  
    mean_values = [r["mean_value"] for r in all_results]
    print(f"   Средние значения: {np.mean(mean_values):.3f}±{np.std(mean_values):.3f}")

# Итоговая оценка
print(f"\n🎯 ОЦЕНКА ДАННЫХ:")

data_issues = []
if len(all_results) == 0:
    data_issues.append("Не удалось загрузить файлы")
elif shape_ok_count < len(all_results):
    data_issues.append(f"Неправильные размерности в {len(all_results)-shape_ok_count} файлах")
elif range_ok_count < len(all_results):
    data_issues.append(f"Неправильный диапазон значений в {len(all_results)-range_ok_count} файлах")
elif no_nan_count < len(all_results) or no_inf_count < len(all_results):
    data_issues.append("Обнаружены NaN или Inf значения")

if len(data_issues) == 0:
    print("   ✅ ВСЕ ДАННЫЕ КОРРЕКТНЫ")
    print("   🚀 Готовы к загрузке на GPU")
else:
    print("   ❌ ОБНАРУЖЕНЫ ПРОБЛЕМЫ:")
    for issue in data_issues:
        print(f"       - {issue}")
    print("   ⚠️ Исправьте перед запуском на GPU")

print(f"\n💾 Результаты проверки сохранены в переменной 'data_check_results'")
data_check_results = {
    "normal_files": normal_check_results,
    "pathology_files": pathology_check_results,
    "total_checked": len(all_results),
    "issues_found": data_issues
}

🔍 ПРОВЕРКА ЦЕЛОСТНОСТИ ДАННЫХ
🟢 ПРОВЕРКА НОРМАЛЬНЫХ ФАЙЛОВ:
   [1/5] Проверяем: f49fd7e1.nii.gz
       Shape: (128, 128, 64) ✅
       Range: [0.000, 1.000] ✅
       Mean±Std: 0.392±0.335
       Size: 2.9 MB
       No NaN/Inf: ✅
   [2/5] Проверяем: c56a338f.nii.gz
       Shape: (128, 128, 64) ✅
       Range: [0.000, 1.000] ✅
       Mean±Std: 0.271±0.334
       Size: 2.9 MB
       No NaN/Inf: ✅
   [3/5] Проверяем: 04e13916.nii.gz
       Shape: (128, 128, 64) ✅
       Range: [0.000, 1.000] ✅
       Mean±Std: 0.260±0.319
       Size: 2.5 MB
       No NaN/Inf: ✅
   [4/5] Проверяем: b9bb2a21.nii.gz
       Shape: (128, 128, 64) ✅
       Range: [0.000, 1.000] ✅
       Mean±Std: 0.409±0.329
       Size: 2.9 MB
       No NaN/Inf: ✅
   [5/5] Проверяем: 0535ca0f.nii.gz
       Shape: (128, 128, 64) ✅
       Range: [0.000, 1.000] ✅
       Mean±Std: 0.354±0.337
       Size: 2.9 MB
       No NaN/Inf: ✅

🔴 ПРОВЕРКА ПАТОЛОГИЧЕСКИХ ФАЙЛОВ:
   [1/5] Проверяем: 70a97d85.nii.gz
       Shape: (128, 128, 64) 

## 2.3: MedVAE model loading test

Что мы делаем:

Поскольку MedVAE не импортировался на CPU машине, создадим mock версию для тестирования всего pipeline. На GPU машине мы заменим на реальную модель.

In [10]:
# # ===================================
# # ЯЧЕЙКА 2.3: MEDVAE MODEL LOADING TEST
# # ===================================

# print("🏥 ТЕСТИРОВАНИЕ MEDVAE MODEL LOADING")
# print("=" * 60)

# # Поскольку MedVAE не импортировался, создаем Mock класс для тестирования
# print("📋 Создаем Mock MedVAE для CPU тестирования...")

# import torch
# import torch.nn as nn

# class MockMedVAE3D(nn.Module):
#     """Mock версия MedVAE3D для CPU тестирования pipeline"""
    
#     def __init__(self, input_shape=(1, 128, 128, 64)):
#         super().__init__()
#         self.input_shape = input_shape
        
#         # Простой 3D autoencoder для тестирования
#         # Encoder
#         self.encoder = nn.Sequential(
#             nn.Conv3d(1, 32, kernel_size=4, stride=2, padding=1),  # -> 64x64x32
#             nn.ReLU(),
#             nn.Conv3d(32, 64, kernel_size=4, stride=2, padding=1), # -> 32x32x16  
#             nn.ReLU(),
#             nn.Conv3d(64, 128, kernel_size=4, stride=2, padding=1), # -> 16x16x8
#             nn.ReLU()
#         )
        
#         # Decoder
#         self.decoder = nn.Sequential(
#             nn.ConvTranspose3d(128, 64, kernel_size=4, stride=2, padding=1), # -> 32x32x16
#             nn.ReLU(),
#             nn.ConvTranspose3d(64, 32, kernel_size=4, stride=2, padding=1),  # -> 64x64x32
#             nn.ReLU(), 
#             nn.ConvTranspose3d(32, 1, kernel_size=4, stride=2, padding=1),   # -> 128x128x64
#             nn.Sigmoid()  # Выход в диапазоне [0,1]
#         )
        
#         print(f"   ✅ Mock MedVAE3D создан")
#         print(f"   📐 Input shape: {input_shape}")
        
#         # Подсчитываем параметры
#         total_params = sum(p.numel() for p in self.parameters())
#         print(f"   🔧 Параметров: {total_params:,}")
    
#     def forward(self, x):
#         encoded = self.encoder(x)
#         reconstructed = self.decoder(encoded)
#         return reconstructed
    
#     @classmethod
#     def from_pretrained(cls, model_name="compression_64"):
#         """Имитирует загрузку предобученной модели"""
#         print(f"   📥 [MOCK] Загружаем модель: {model_name}")
#         return cls()

# # Тестируем создание модели
# print("\n🧪 ТЕСТИРОВАНИЕ СОЗДАНИЯ МОДЕЛИ:")

# try:
#     # Создаем mock модель
#     mock_model = MockMedVAE3D.from_pretrained("compression_64")
#     print("   ✅ Модель создана успешно")
    
#     # Проверяем на CPU device
#     device = torch.device('cpu')  # Принудительно CPU для тестирования
#     mock_model = mock_model.to(device)
#     mock_model.eval()
    
#     print(f"   📱 Модель перемещена на device: {device}")
    
# except Exception as e:
#     print(f"   ❌ Ошибка создания модели: {e}")
#     mock_model = None

# # Тестируем forward pass
# print("\n🔄 ТЕСТИРОВАНИЕ FORWARD PASS:")

# if mock_model is not None:
#     try:
#         # Создаем dummy input
#         batch_size = 2
#         dummy_input = torch.randn(batch_size, 1, 128, 128, 64)
#         print(f"   📊 Dummy input shape: {dummy_input.shape}")
        
#         # Forward pass
#         with torch.no_grad():
#             start_time = time.time()
#             dummy_output = mock_model(dummy_input)
#             forward_time = time.time() - start_time
            
#         print(f"   ✅ Forward pass успешен")
#         print(f"   📊 Output shape: {dummy_output.shape}")
#         print(f"   ⏱️ Время forward pass: {forward_time:.3f} сек")
#         print(f"   📈 Output range: [{dummy_output.min():.3f}, {dummy_output.max():.3f}]")
        
#         # Проверяем что output в правильном диапазоне
#         output_in_range = (dummy_output >= 0) & (dummy_output <= 1)
#         if output_in_range.all():
#             print(f"   ✅ Output в диапазоне [0,1]")
#         else:
#             print(f"   ⚠️ Output вне диапазона [0,1]")
            
#     except Exception as e:
#         print(f"   ❌ Ошибка forward pass: {e}")
#         dummy_output = None
# else:
#     dummy_output = None

# # Тестируем расчет reconstruction loss
# print("\n📊 ТЕСТИРОВАНИЕ RECONSTRUCTION LOSS:")

# if dummy_output is not None:
#     try:
#         # MSE Loss
#         mse_loss = nn.MSELoss()
#         loss_value = mse_loss(dummy_output, dummy_input)
        
#         print(f"   ✅ MSE Loss вычислен: {loss_value.item():.6f}")
        
#         # L1 Loss для сравнения
#         l1_loss = nn.L1Loss()
#         l1_value = l1_loss(dummy_output, dummy_input)
        
#         print(f"   ℹ️ L1 Loss для сравнения: {l1_value.item():.6f}")
        
#     except Exception as e:
#         print(f"   ❌ Ошибка loss calculation: {e}")

# # Тестируем сохранение/загрузку модели
# print("\n💾 ТЕСТИРОВАНИЕ CHECKPOINT OPERATIONS:")

# if mock_model is not None:
#     try:
#         # Сохраняем checkpoint
#         checkpoint = {
#             'model_state_dict': mock_model.state_dict(),
#             'model_name': 'mock_medvae_3d',
#             'input_shape': (1, 128, 128, 64)
#         }
        
#         checkpoint_path = "test_mock_checkpoint.pth"
#         torch.save(checkpoint, checkpoint_path)
        
#         print(f"   ✅ Checkpoint сохранен: {checkpoint_path}")
        
#         # Загружаем checkpoint
#         loaded_checkpoint = torch.load(checkpoint_path, map_location='cpu')
        
#         # Создаем новую модель и загружаем веса
#         test_model = MockMedVAE3D()
#         test_model.load_state_dict(loaded_checkpoint['model_state_dict'])
        
#         print(f"   ✅ Checkpoint загружен успешно")
        
#         # Удаляем тестовый файл
#         import os
#         os.remove(checkpoint_path)
#         print(f"   🗑️ Тестовый checkpoint удален")
        
#     except Exception as e:
#         print(f"   ❌ Ошибка checkpoint operations: {e}")

# # Итоговая оценка
# print(f"\n🎯 ОЦЕНКА MODEL LOADING TEST:")

# model_tests = [
#     ("Создание модели", mock_model is not None),
#     ("Forward pass", dummy_output is not None),
#     ("Loss calculation", dummy_output is not None),
#     ("Checkpoint operations", True)  # Если дошли сюда, то ОК
# ]

# all_tests_passed = all(test_result for _, test_result in model_tests)

# for test_name, test_result in model_tests:
#     status = "✅" if test_result else "❌"
#     print(f"   {status} {test_name}")

# if all_tests_passed:
#     print(f"\n✅ ВСЕ ТЕСТЫ МОДЕЛИ ПРОШЛИ")
#     print(f"   🚀 Pipeline готов к работе с реальной MedVAE на GPU")
# else:
#     print(f"\n❌ ЕСТЬ ПРОБЛЕМЫ В ТЕСТАХ МОДЕЛИ")
#     print(f"   ⚠️ Проверьте ошибки перед GPU запуском")

# print(f"\n💡 ЗАМЕТКА: На GPU машине замените MockMedVAE3D на реальную MedVAE")


🏥 ТЕСТИРОВАНИЕ MEDVAE MODEL LOADING
📋 Создаем Mock MedVAE для CPU тестирования...

🧪 ТЕСТИРОВАНИЕ СОЗДАНИЯ МОДЕЛИ:
   📥 [MOCK] Загружаем модель: compression_64
   ✅ Mock MedVAE3D создан
   📐 Input shape: (1, 128, 128, 64)
   🔧 Параметров: 1,315,137
   ✅ Модель создана успешно
   📱 Модель перемещена на device: cpu

🔄 ТЕСТИРОВАНИЕ FORWARD PASS:
   📊 Dummy input shape: torch.Size([2, 1, 128, 128, 64])
   ✅ Forward pass успешен
   📊 Output shape: torch.Size([2, 1, 128, 128, 64])
   ⏱️ Время forward pass: 0.822 сек
   📈 Output range: [0.512, 0.522]
   ✅ Output в диапазоне [0,1]

📊 ТЕСТИРОВАНИЕ RECONSTRUCTION LOSS:
   ✅ MSE Loss вычислен: 1.268254
   ℹ️ L1 Loss для сравнения: 0.902536

💾 ТЕСТИРОВАНИЕ CHECKPOINT OPERATIONS:
   ✅ Checkpoint сохранен: test_mock_checkpoint.pth
   ✅ Mock MedVAE3D создан
   📐 Input shape: (1, 128, 128, 64)
   🔧 Параметров: 1,315,137
   ✅ Checkpoint загружен успешно
   🗑️ Тестовый checkpoint удален

🎯 ОЦЕНКА MODEL LOADING TEST:
   ✅ Создание модели
   ✅ Forward pas

## ЭТАП 2.4: Functions dry run

Что мы делаем:
Тестируем все utility функции и метрики на fake данных, чтобы убедиться что весь workflow работает перед GPU запуском.

In [11]:
# ===================================
# ЯЧЕЙКА 2.4: FUNCTIONS DRY RUN
# ===================================

print("⚙️ ТЕСТИРОВАНИЕ UTILITY ФУНКЦИЙ")
print("=" * 60)

import time
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, classification_report

# Создаем mock данные для тестирования
print("📊 СОЗДАНИЕ MOCK ДАННЫХ ДЛЯ ТЕСТИРОВАНИЯ:")

# Mock reconstruction errors
np.random.seed(42)  # Для воспроизводимости

# Нормальные данные должны иметь низкие reconstruction errors
normal_errors = np.random.normal(0.001, 0.0002, 50)  # Mean=0.001, std=0.0002
normal_errors = np.clip(normal_errors, 0, None)  # Только положительные

# Патологии должны иметь более высокие reconstruction errors
pathology_errors = np.random.normal(0.0025, 0.0008, 50)  # Mean=0.0025, std=0.0008  
pathology_errors = np.clip(pathology_errors, 0, None)

print(f"   📈 Normal errors: mean={normal_errors.mean():.6f}, std={normal_errors.std():.6f}")
print(f"   📈 Pathology errors: mean={pathology_errors.mean():.6f}, std={pathology_errors.std():.6f}")

# Объединяем для mixed test set
all_errors = np.concatenate([normal_errors, pathology_errors])
ground_truth = np.concatenate([np.zeros(50), np.ones(50)])  # 0=normal, 1=pathology

print(f"   📊 Test set: {len(all_errors)} samples (50 normal, 50 pathology)")

# ФУНКЦИЯ 1: Anomaly score computation
print(f"\n🧮 ТЕСТИРОВАНИЕ: Anomaly score computation")

def compute_anomaly_scores_test(normal_errors, test_errors):
    """Вычисляет anomaly scores на основе статистики нормальных данных"""
    
    # Статистика нормальных данных  
    normal_mean = np.mean(normal_errors)
    normal_std = np.std(normal_errors)
    
    print(f"   📊 Normal statistics: mean={normal_mean:.6f}, std={normal_std:.6f}")
    
    # Threshold: mean + 2*std
    threshold = normal_mean + 2 * normal_std
    print(f"   🎯 Anomaly threshold: {threshold:.6f}")
    
    # Z-score based anomaly scores
    anomaly_scores = (test_errors - normal_mean) / normal_std
    
    return anomaly_scores, threshold

try:
    # Используем первые 30 normal errors для "training" статистики
    train_normal_errors = normal_errors[:30]
    
    anomaly_scores, threshold = compute_anomaly_scores_test(train_normal_errors, all_errors)
    print(f"   ✅ Anomaly scores computed: shape={anomaly_scores.shape}")
    print(f"   📈 Score range: [{anomaly_scores.min():.3f}, {anomaly_scores.max():.3f}]")
    
except Exception as e:
    print(f"   ❌ Ошибка anomaly score computation: {e}")
    anomaly_scores = None

# ФУНКЦИЯ 2: Metrics computation
print(f"\n📊 ТЕСТИРОВАНИЕ: Metrics computation")

if anomaly_scores is not None:
    try:
        # ROC AUC
        auc_score = roc_auc_score(ground_truth, anomaly_scores)
        print(f"   🎯 ROC AUC: {auc_score:.4f}")
        
        # ROC Curve
        fpr, tpr, roc_thresholds = roc_curve(ground_truth, anomaly_scores)
        print(f"   📈 ROC curve points: {len(fpr)}")
        
        # Precision-Recall curve
        precision, recall, pr_thresholds = precision_recall_curve(ground_truth, anomaly_scores)
        print(f"   📈 PR curve points: {len(precision)}")
        
        # Optimal threshold (maximize F1-score)
        f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
        best_f1_idx = np.argmax(f1_scores)
        best_f1 = f1_scores[best_f1_idx]
        best_threshold = pr_thresholds[best_f1_idx] if best_f1_idx < len(pr_thresholds) else threshold
        
        print(f"   🏆 Best F1-score: {best_f1:.4f} at threshold: {best_threshold:.4f}")
        
        # Classification report
        y_pred = (anomaly_scores > best_threshold).astype(int)
        class_report = classification_report(ground_truth, y_pred, target_names=['Normal', 'Pathology'], output_dict=True)
        
        print(f"   📋 Classification metrics:")
        print(f"       Normal: precision={class_report['Normal']['precision']:.3f}, recall={class_report['Normal']['recall']:.3f}")
        print(f"       Pathology: precision={class_report['Pathology']['precision']:.3f}, recall={class_report['Pathology']['recall']:.3f}")
        
        print(f"   ✅ All metrics computed successfully")
        
    except Exception as e:
        print(f"   ❌ Ошибка metrics computation: {e}")

# ФУНКЦИЯ 3: Dataset и DataLoader тестирование
print(f"\n📁 ТЕСТИРОВАНИЕ: Dataset/DataLoader simulation")

from torch.utils.data import Dataset, DataLoader

class MockCTDataset(Dataset):
    """Mock dataset для тестирования"""
    
    def __init__(self, file_list, data_shape=(128, 128, 64)):
        self.file_list = file_list
        self.data_shape = data_shape
        
    def __len__(self):
        return len(self.file_list)
    
    def __getitem__(self, idx):
        filename = self.file_list[idx]
        
        # Создаем fake volume
        volume = torch.randn(1, *self.data_shape) * 0.1 + 0.3  # Mean~0.3, std~0.1
        volume = torch.clamp(volume, 0, 1)  # Clamp to [0,1]
        
        return {
            'image': volume,
            'filename': filename,
            'idx': idx
        }

try:
    # Создаем mock dataset
    mock_files = [f"mock_{i:03d}.nii.gz" for i in range(10)]
    mock_dataset = MockCTDataset(mock_files)
    
    print(f"   📊 Mock dataset: {len(mock_dataset)} files")
    
    # Тестируем DataLoader
    mock_loader = DataLoader(mock_dataset, batch_size=3, shuffle=True, num_workers=0)
    
    print(f"   🔄 Mock DataLoader: batch_size=3, batches={len(mock_loader)}")
    
    # Тестируем загрузку одного batch
    start_time = time.time()
    for i, batch in enumerate(mock_loader):
        if i == 0:  # Только первый batch
            print(f"   📊 Batch shape: {batch['image'].shape}")
            print(f"   📁 Batch files: {batch['filename']}")
            print(f"   📈 Batch range: [{batch['image'].min():.3f}, {batch['image'].max():.3f}]")
        break
    
    load_time = time.time() - start_time
    print(f"   ⏱️ Batch load time: {load_time:.3f} seconds")
    print(f"   ✅ DataLoader test passed")
    
except Exception as e:
    print(f"   ❌ Ошибка DataLoader test: {e}")

# ФУНКЦИЯ 4: результат saving/loading
print(f"\n💾 ТЕСТИРОВАНИЕ: Results saving/loading")

try:
    # Создаем mock results
    mock_results = {
        'model_name': 'MedVAE_3D_baseline',
        'auc_score': float(auc_score) if 'auc_score' in locals() else 0.85,
        'best_f1': float(best_f1) if 'best_f1' in locals() else 0.75,
        'best_threshold': float(best_threshold) if 'best_threshold' in locals() else 2.0,
        'test_files': mock_files,
        'anomaly_scores': anomaly_scores.tolist() if anomaly_scores is not None else [],
        'ground_truth': ground_truth.tolist(),
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
    }
    
    # Сохраняем в JSON
    results_file = "test_results.json"
    
    import json
    with open(results_file, 'w') as f:
        json.dump(mock_results, f, indent=2)
    
    print(f"   ✅ Results saved to: {results_file}")
    
    # Загружаем обратно
    with open(results_file, 'r') as f:
        loaded_results = json.load(f)
    
    print(f"   ✅ Results loaded successfully")
    print(f"   📊 Loaded AUC: {loaded_results['auc_score']:.4f}")
    
    # Удаляем тестовый файл
    import os
    os.remove(results_file)
    print(f"   🗑️ Test file cleaned up")
    
except Exception as e:
    print(f"   ❌ Ошибка results saving/loading: {e}")

# Итоговая оценка всех функций
print(f"\n🎯 ОЦЕНКА FUNCTIONS DRY RUN:")

function_tests = [
    ("Anomaly score computation", anomaly_scores is not None),
    ("Metrics computation", 'auc_score' in locals()),
    ("DataLoader simulation", True),  # Если дошли до сюда
    ("Results saving/loading", True)   # Если дошли до сюда
]

all_functions_ok = all(test_result for _, test_result in function_tests)

for test_name, test_result in function_tests:
    status = "✅" if test_result else "❌"
    print(f"   {status} {test_name}")

if all_functions_ok:
    print(f"\n✅ ВСЕ ФУНКЦИИ ПРОТЕСТИРОВАНЫ УСПЕШНО")
    print(f"   🚀 Pipeline полностью готов к GPU запуску")
else:
    print(f"\n❌ ЕСТЬ ПРОБЛЕМЫ В ФУНКЦИЯХ")
    print(f"   ⚠️ Исправьте перед GPU запуском")

print(f"\n🎉 ЭТАП 2: CPU ТЕСТИРОВАНИЕ ЗАВЕРШЕН")
print(f"   ✅ Все компоненты pipeline протестированы")
print(f"   🎯 Готовы к переходу на GPU для реального обучения")


⚙️ ТЕСТИРОВАНИЕ UTILITY ФУНКЦИЙ
📊 СОЗДАНИЕ MOCK ДАННЫХ ДЛЯ ТЕСТИРОВАНИЯ:
   📈 Normal errors: mean=0.000955, std=0.000185
   📈 Pathology errors: mean=0.002514, std=0.000692
   📊 Test set: 100 samples (50 normal, 50 pathology)

🧮 ТЕСТИРОВАНИЕ: Anomaly score computation
   📊 Normal statistics: mean=0.000962, std=0.000177
   🎯 Anomaly threshold: 0.001316
   ✅ Anomaly scores computed: shape=(100,)
   📈 Score range: [-3.154, 15.761]

📊 ТЕСТИРОВАНИЕ: Metrics computation
   🎯 ROC AUC: 0.9684
   📈 ROC curve points: 9
   📈 PR curve points: 101
   🏆 Best F1-score: 0.9697 at threshold: 2.0727
   📋 Classification metrics:
       Normal: precision=0.942, recall=0.980
       Pathology: precision=0.979, recall=0.940
   ✅ All metrics computed successfully

📁 ТЕСТИРОВАНИЕ: Dataset/DataLoader simulation
   📊 Mock dataset: 10 files
   🔄 Mock DataLoader: batch_size=3, batches=4
   📊 Batch shape: torch.Size([3, 1, 128, 128, 64])
   📁 Batch files: ['mock_005.nii.gz', 'mock_000.nii.gz', 'mock_003.nii.gz']
   

# ЭТАП 3: DATASET PREPARATION & LOADING

## 3.1: Custom Dataset класс implementation
Что мы делаем:
Создаем production-ready Dataset класс для загрузки наших предобработанных .nii.gz файлов с поддержкой train/val/test splits и правильным error handling.

In [12]:
print("📁 СОЗДАНИЕ PRODUCTION-READY DATASET CLASS")
print("=" * 60)

class CTAnomalyDataset(Dataset):
    """
    Dataset для загрузки предобработанных КТ томов для anomaly detection
    
    Features:
    - Поддержка train/validation/test splits
    - Error handling для поврежденных файлов  
    - Гибкая конфигурация путей к данным
    - Опциональные transforms
    """
    
    def __init__(self, 
                 normal_data_path,
                 pathology_data_path, 
                 file_list,
                 split_type='train',
                 ground_truth_labels=None,
                 transform=None,
                 expected_shape=(128, 128, 64),
                 cache_in_memory=False):
        """
        Args:
            normal_data_path: Path к папке с нормальными КТ
            pathology_data_path: Path к папке с патологическими КТ  
            file_list: Список файлов для этого split
            split_type: 'train', 'validation', или 'test'
            ground_truth_labels: Словарь {filename: label}
            transform: Опциональные transforms  
            expected_shape: Ожидаемая размерность тензора
            cache_in_memory: Кэшировать данные в RAM (для малых датасетов)
        """
        
        self.normal_data_path = Path(normal_data_path)
        self.pathology_data_path = Path(pathology_data_path)
        self.file_list = file_list
        self.split_type = split_type
        self.ground_truth_labels = ground_truth_labels or {}
        self.transform = transform
        self.expected_shape = expected_shape
        self.cache_in_memory = cache_in_memory
        
        # Кэш для данных в памяти
        self.memory_cache = {} if cache_in_memory else None
        
        # Валидируем пути
        self._validate_paths()
        
        # Создаем mapping filename -> full_path
        self.file_paths = self._create_file_mapping()
        
        # Статистика
        self._compute_split_stats()
        
        print(f"✅ CTAnomalyDataset создан:")
        print(f"   Split: {split_type}")
        print(f"   Files: {len(self.file_list)}")
        print(f"   Normal files: {self.normal_count}")
        print(f"   Pathology files: {self.pathology_count}")
        print(f"   Memory cache: {'Enabled' if cache_in_memory else 'Disabled'}")
        
    def _validate_paths(self):
        """Проверяет существование путей к данным"""
        if not self.normal_data_path.exists():
            raise ValueError(f"Normal data path не существует: {self.normal_data_path}")
        if not self.pathology_data_path.exists():
            raise ValueError(f"Pathology data path не существует: {self.pathology_data_path}")
    
    def _create_file_mapping(self):
        """Создает mapping filename -> full file path"""
        file_paths = {}
        
        for filename in self.file_list:
            # Ищем файл в нормальной папке
            normal_path = self.normal_data_path / filename
            pathology_path = self.pathology_data_path / filename
            
            if normal_path.exists():
                file_paths[filename] = normal_path
            elif pathology_path.exists():
                file_paths[filename] = pathology_path
            else:
                print(f"⚠️ Файл не найден: {filename}")
                # Можно либо пропустить, либо raise error
                continue
                
        return file_paths
    
    def _compute_split_stats(self):
        """Вычисляет статистику по split"""
        self.normal_count = 0
        self.pathology_count = 0
        
        for filename in self.file_list:
            if filename in self.file_paths:  # Только существующие файлы
                label = self.ground_truth_labels.get(filename, 0)
                if label == 0:
                    self.normal_count += 1
                else:
                    self.pathology_count += 1
    
    def __len__(self):
        return len([f for f in self.file_list if f in self.file_paths])
    
    def __getitem__(self, idx):
        """Загружает один КТ том"""
        
        # Получаем filename (только среди существующих файлов)
        available_files = [f for f in self.file_list if f in self.file_paths]
        
        if idx >= len(available_files):
            raise IndexError(f"Index {idx} out of range for {len(available_files)} files")
            
        filename = available_files[idx]
        
        # Проверяем кэш
        if self.cache_in_memory and filename in self.memory_cache:
            volume = self.memory_cache[filename]
        else:
            # Загружаем файл
            volume = self._load_volume(filename)
            
            # Кэшируем если включено
            if self.cache_in_memory:
                self.memory_cache[filename] = volume.clone()
        
        # Применяем transforms если есть
        if self.transform:
            volume = self.transform(volume)
        
        # Получаем label и дополнительную информацию
        label = self.ground_truth_labels.get(filename, 0)
        file_path = self.file_paths[filename]
        
        return {
            'image': volume,           # Tensor (1, H, W, D)
            'label': label,            # 0 or 1
            'filename': filename,      # str
            'filepath': str(file_path), # str  
            'idx': idx,                # int
            'split': self.split_type   # str
        }

    def _load_volume(self, filename):
        """Загружает один .nii.gz файл с улучшенным error handling"""
        file_path = self.file_paths[filename]
        
        try:
            # Загружаем с помощью nibabel
            nii_img = nib.load(str(file_path))
            volume = nii_img.get_fdata().astype(np.float32)
            
            # Проверяем размерность
            if volume.shape != self.expected_shape:
                print(f"⚠️ Неожиданная размерность {filename}: {volume.shape}, ожидали {self.expected_shape}")
                # Добавляем файл в список проблемных
                if not hasattr(self, 'problematic_files'):
                    self.problematic_files = []
                self.problematic_files.append(filename)
            
            # Проверяем диапазон значений
            if volume.min() < -0.1 or volume.max() > 1.1:
                print(f"⚠️ Значения вне диапазона [0,1] в {filename}: [{volume.min():.3f}, {volume.max():.3f}]")
                if not hasattr(self, 'problematic_files'):
                    self.problematic_files = []
                self.problematic_files.append(filename)
            
            # Проверяем на NaN/Inf
            if np.isnan(volume).any() or np.isinf(volume).any():
                print(f"❌ Обнаружены NaN/Inf в {filename} - файл пропускается")
                if not hasattr(self, 'problematic_files'):
                    self.problematic_files = []
                self.problematic_files.append(filename)
                raise ValueError(f"NaN/Inf values in {filename}")
            
            # Преобразуем в PyTorch tensor и добавляем channel dimension
            volume_tensor = torch.from_numpy(volume).unsqueeze(0)  # (H,W,D) -> (1,H,W,D)
            
            return volume_tensor
            
        except Exception as e:
            print(f"❌ Ошибка загрузки {filename}: {e}")
            # Добавляем в список проблемных файлов
            if not hasattr(self, 'problematic_files'):
                self.problematic_files = []
            self.problematic_files.append(filename)
            
            # Возвращаем None вместо zero tensor - будем skip в DataLoader
            return None
    
    def get_problematic_files(self):
        """Возвращает список проблемных файлов"""
        return getattr(self, 'problematic_files', [])
    
    def get_split_info(self):
        """Возвращает подробную информацию о split"""
        return {
            'split_type': self.split_type,
            'total_files': len(self),
            'normal_count': self.normal_count,
            'pathology_count': self.pathology_count,
            'normal_percentage': self.normal_count / len(self) * 100 if len(self) > 0 else 0,
            'pathology_percentage': self.pathology_count / len(self) * 100 if len(self) > 0 else 0,
            'cache_enabled': self.cache_in_memory,
            'expected_shape': self.expected_shape
        }

print("✅ CTAnomalyDataset class определен")

# Тестируем создание dataset instances для каждого split
print("\n🧪 ТЕСТИРОВАНИЕ СОЗДАНИЯ DATASETS:")

try:
    # Training dataset (только нормы)
    print("\n📚 Создаем TRAINING dataset...")
    train_dataset = CTAnomalyDataset(
        normal_data_path=NORMAL_DATA_PATH,
        pathology_data_path=PATHOLOGY_DATA_PATH,
        file_list=train_files,
        split_type='train', 
        ground_truth_labels=ground_truth_labels,
        expected_shape=(128, 128, 64),
        cache_in_memory=False  # Слишком много для кэша
    )
    
    print(f"✅ Training dataset создан: {len(train_dataset)} файлов")
    
    # Validation dataset (нормы + патологии)
    print("\n🔍 Создаем VALIDATION dataset...")
    val_dataset = CTAnomalyDataset(
        normal_data_path=NORMAL_DATA_PATH,
        pathology_data_path=PATHOLOGY_DATA_PATH,
        file_list=val_files,
        split_type='validation',
        ground_truth_labels=ground_truth_labels,
        expected_shape=(128, 128, 64),
        cache_in_memory=True   # Меньше файлов, можно кэшировать
    )
    
    print(f"✅ Validation dataset создан: {len(val_dataset)} файлов")
    
    # Test dataset (нормы + патологии)
    print("\n🧪 Создаем TEST dataset...")
    test_dataset = CTAnomalyDataset(
        normal_data_path=NORMAL_DATA_PATH,
        pathology_data_path=PATHOLOGY_DATA_PATH,
        file_list=test_files,
        split_type='test',
        ground_truth_labels=ground_truth_labels, 
        expected_shape=(128, 128, 64),
        cache_in_memory=False  # Средний размер, без кэша
    )
    
    print(f"✅ Test dataset создан: {len(test_dataset)} файлов")
    
except Exception as e:
    print(f"❌ Ошибка создания datasets: {e}")
    train_dataset = val_dataset = test_dataset = None

# Выводим статистику по всем datasets
if train_dataset and val_dataset and test_dataset:
    print(f"\n📊 СТАТИСТИКА DATASETS:")
    
    for dataset_name, dataset in [('Train', train_dataset), ('Validation', val_dataset), ('Test', test_dataset)]:
        info = dataset.get_split_info()
        print(f"\n🔹 {dataset_name.upper()}:")
        print(f"   Всего файлов: {info['total_files']}")
        print(f"   Нормальные: {info['normal_count']} ({info['normal_percentage']:.1f}%)")
        print(f"   Патологические: {info['pathology_count']} ({info['pathology_percentage']:.1f}%)")
        print(f"   Кэш в памяти: {info['cache_enabled']}")

print(f"\n🎯 DATASET CLASSES ГОТОВЫ")
print(f"   ✅ Все три split созданы успешно")  
print(f"   🚀 Готовы к созданию DataLoaders")


📁 СОЗДАНИЕ PRODUCTION-READY DATASET CLASS
✅ CTAnomalyDataset class определен

🧪 ТЕСТИРОВАНИЕ СОЗДАНИЯ DATASETS:

📚 Создаем TRAINING dataset...
✅ CTAnomalyDataset создан:
   Split: train
   Files: 313
   Normal files: 313
   Pathology files: 0
   Memory cache: Disabled
✅ Training dataset создан: 313 файлов

🔍 Создаем VALIDATION dataset...
✅ CTAnomalyDataset создан:
   Split: validation
   Files: 229
   Normal files: 43
   Pathology files: 186
   Memory cache: Enabled
✅ Validation dataset создан: 229 файлов

🧪 Создаем TEST dataset...
✅ CTAnomalyDataset создан:
   Split: test
   Files: 249
   Normal files: 63
   Pathology files: 186
   Memory cache: Disabled
✅ Test dataset создан: 249 файлов

📊 СТАТИСТИКА DATASETS:

🔹 TRAIN:
   Всего файлов: 313
   Нормальные: 313 (100.0%)
   Патологические: 0 (0.0%)
   Кэш в памяти: False

🔹 VALIDATION:
   Всего файлов: 229
   Нормальные: 43 (18.8%)
   Патологические: 186 (81.2%)
   Кэш в памяти: True

🔹 TEST:
   Всего файлов: 249
   Нормальные: 63 (25.3

## 3.2: DataLoader configuration
Что мы делаем:
Создаем оптимизированные DataLoader для каждого split с настройками под V100 GPU и нашу задачу anomaly detection.

In [13]:
print("🔄 КОНФИГУРАЦИЯ DATALOADERS")
print("=" * 60)

# Решение multiprocessing проблемы: используем num_workers=0 для тестирования на CPU
# На GPU будем использовать standard collate без кастомной функции

print("🔧 Multiprocessing: отключен для CPU тестирования (включится на GPU)")

# V100 оптимизированные конфигурации (адаптированные для CPU тестирования)
print("\n⚙️ КОНФИГУРАЦИЯ ДЛЯ V100 GPU (CPU тест версия):")

# Training DataLoader (только нормы, нужен shuffle)
TRAIN_CONFIG = {
    'batch_size': 3,          # V100 16GB может handle 3 тома (128³)
    'shuffle': True,          # Важно для training
    'num_workers': 0,         # На CPU: 0, на GPU: 4
    'pin_memory': False,      # На CPU: False, на GPU: True  
    'drop_last': True,        # Consistent batch sizes
    # 'collate_fn': стандартная будет использована
}

# Validation DataLoader (нормы + патологии, no shuffle)  
VAL_CONFIG = {
    'batch_size': 2,          # Чуть меньше для стабильности
    'shuffle': False,         # Deterministic validation
    'num_workers': 0,         # На CPU: 0, на GPU: 3
    'pin_memory': False,      # На CPU: False, на GPU: True
    'drop_last': False,       # Используем все validation данные
}

# Test DataLoader (нормы + патологии, single samples для точности)
TEST_CONFIG = {
    'batch_size': 1,          # По одному для точных метрик
    'shuffle': False,         # Deterministic evaluation
    'num_workers': 0,         # На CPU: 0, на GPU: 2
    'pin_memory': False,      # На CPU: False, на GPU: True
    'drop_last': False,       # Важно не терять test данные
}

# GPU конфигурации для справки (будем использовать на GPU)
GPU_TRAIN_CONFIG = {
    'batch_size': 3,
    'shuffle': True,
    'num_workers': 4,
    'pin_memory': True,
    'persistent_workers': True,
    'prefetch_factor': 2,
    'drop_last': True,
    'timeout': 60,
}

GPU_VAL_CONFIG = {
    'batch_size': 2,
    'shuffle': False,
    'num_workers': 3,
    'pin_memory': True,
    'persistent_workers': True,
    'prefetch_factor': 2,
    'drop_last': False,
    'timeout': 60,
}

GPU_TEST_CONFIG = {
    'batch_size': 1,
    'shuffle': False,
    'num_workers': 2,
    'pin_memory': True,
    'drop_last': False,
    'timeout': 30,
}

print(f"📊 CPU TEST КОНФИГУРАЦИИ:")
print(f"   Training: batch_size={TRAIN_CONFIG['batch_size']}, workers={TRAIN_CONFIG['num_workers']}")
print(f"   Validation: batch_size={VAL_CONFIG['batch_size']}, workers={VAL_CONFIG['num_workers']}")  
print(f"   Test: batch_size={TEST_CONFIG['batch_size']}, workers={TEST_CONFIG['num_workers']}")

print(f"🔥 GPU FINAL КОНФИГУРАЦИИ (будут использованы на GPU):")
print(f"   Training: batch_size={GPU_TRAIN_CONFIG['batch_size']}, workers={GPU_TRAIN_CONFIG['num_workers']}")
print(f"   Validation: batch_size={GPU_VAL_CONFIG['batch_size']}, workers={GPU_VAL_CONFIG['num_workers']}")  
print(f"   Test: batch_size={GPU_TEST_CONFIG['batch_size']}, workers={GPU_TEST_CONFIG['num_workers']}")

# Создаем простые mock datasets без кастомной collate
print(f"\n🔧 СОЗДАНИЕ ПРОСТЫХ DATALOADERS ДЛЯ CPU ТЕСТА:")

from torch.utils.data import TensorDataset

# Mock datasets для тестирования
train_images = torch.randn(9, 1, 128, 128, 64) * 0.1 + 0.3  # 9 samples для 3 batches
train_labels = torch.zeros(9, dtype=torch.long)
mock_train_dataset = TensorDataset(train_images, train_labels)

val_images = torch.randn(6, 1, 128, 128, 64) * 0.1 + 0.3  # 6 samples для 3 batches
val_labels = torch.cat([torch.zeros(3), torch.ones(3)])
mock_val_dataset = TensorDataset(val_images, val_labels)

test_images = torch.randn(5, 1, 128, 128, 64) * 0.1 + 0.3  # 5 samples
test_labels = torch.cat([torch.zeros(2), torch.ones(3)])
mock_test_dataset = TensorDataset(test_images, test_labels)

try:
    # Training DataLoader
    print("\n📚 Создаем Training DataLoader...")
    train_loader = DataLoader(mock_train_dataset, **TRAIN_CONFIG)
    print(f"   ✅ Training DataLoader: {len(train_loader)} batches")
    
    # Validation DataLoader  
    print("🔍 Создаем Validation DataLoader...")
    val_loader = DataLoader(mock_val_dataset, **VAL_CONFIG)
    print(f"   ✅ Validation DataLoader: {len(val_loader)} batches")
    
    # Test DataLoader
    print("🧪 Создаем Test DataLoader...")
    test_loader = DataLoader(mock_test_dataset, **TEST_CONFIG) 
    print(f"   ✅ Test DataLoader: {len(test_loader)} batches")
    
    dataloaders_created = True
    
except Exception as e:
    print(f"❌ Ошибка создания DataLoaders: {e}")
    dataloaders_created = False

# Тестируем загрузку данных (должно работать без multiprocessing)
if dataloaders_created:
    print(f"\n🧪 ТЕСТИРОВАНИЕ ЗАГРУЗКИ ДАННЫХ:")
    
    import time
    
    try:
        # Тест Training DataLoader
        print("📚 Тест Training DataLoader...")
        start_time = time.time()
        
        for i, (images, labels) in enumerate(train_loader):
            if i == 0:  # Только первый batch
                print(f"   📊 Batch shape: {images.shape}")
                print(f"   🏷️ Labels shape: {labels.shape}")
                print(f"   📈 Image range: [{images.min():.3f}, {images.max():.3f}]")
                print(f"   🏷️ Label values: {labels.unique()}")
            break
            
        load_time = time.time() - start_time
        print(f"   ⏱️ Load time: {load_time:.3f}s")
        print(f"   ✅ Training DataLoader test passed")
        
        # Тест Validation DataLoader
        print("\n🔍 Тест Validation DataLoader...")
        val_images, val_labels = next(iter(val_loader))
        print(f"   📊 Val batch shape: {val_images.shape}")
        print(f"   🏷️ Val labels: {val_labels.unique()}")
        print(f"   ✅ Validation DataLoader test passed")
        
        # Тест Test DataLoader
        print("\n🧪 Тест Test DataLoader...")
        test_images, test_labels = next(iter(test_loader))
        print(f"   📊 Test batch shape: {test_images.shape}")
        print(f"   🏷️ Test label: {test_labels}")
        print(f"   ✅ Test DataLoader test passed")
        
        # Тест полного прохода по training loader
        print(f"\n🔄 Тест полного прохода Training DataLoader:")
        total_samples = 0
        for i, (batch_images, batch_labels) in enumerate(train_loader):
            total_samples += len(batch_images)
            print(f"   Batch {i+1}: {len(batch_images)} samples")
        print(f"   📊 Total samples processed: {total_samples}")
        print(f"   ✅ Full training pass test passed")
        
    except Exception as e:
        print(f"❌ Ошибка тестирования DataLoaders: {e}")
        import traceback
        traceback.print_exc()

# Memory estimation остается той же
print(f"\n💾 MEMORY ESTIMATION ДЛЯ V100:")
single_volume_mb = (1 * 128 * 128 * 64 * 4) / (1024 * 1024)  # FP32
print(f"   Single volume: {single_volume_mb:.1f} MB")
print(f"   Training batch (3 volumes): {single_volume_mb * 3:.1f} MB")
print(f"   Validation batch (2 volumes): {single_volume_mb * 2:.1f} MB")
print(f"   Test batch (1 volume): {single_volume_mb:.1f} MB")
print(f"   📊 Total data memory: ~{single_volume_mb * 4:.1f} MB (+ model ~200MB = ~{single_volume_mb * 4 + 200:.0f} MB)")

print(f"\n🎯 DATALOADERS ПРОТЕСТИРОВАНЫ")
print(f"   ✅ Memory footprint приемлемый (~{single_volume_mb * 4 + 200:.0f} MB)")
print(f"   🔥 GPU конфигурации готовы для production use")

# Сохраняем обе конфигурации
dataloader_configs = {
    'cpu_train_config': TRAIN_CONFIG,
    'cpu_val_config': VAL_CONFIG,
    'cpu_test_config': TEST_CONFIG,
    'gpu_train_config': GPU_TRAIN_CONFIG,
    'gpu_val_config': GPU_VAL_CONFIG,
    'gpu_test_config': GPU_TEST_CONFIG,
    'memory_per_volume_mb': single_volume_mb
}

print(f"💾 Обе конфигурации (CPU + GPU) сохранены в 'dataloader_configs'")


🔄 КОНФИГУРАЦИЯ DATALOADERS
🔧 Multiprocessing: отключен для CPU тестирования (включится на GPU)

⚙️ КОНФИГУРАЦИЯ ДЛЯ V100 GPU (CPU тест версия):
📊 CPU TEST КОНФИГУРАЦИИ:
   Training: batch_size=3, workers=0
   Validation: batch_size=2, workers=0
   Test: batch_size=1, workers=0
🔥 GPU FINAL КОНФИГУРАЦИИ (будут использованы на GPU):
   Training: batch_size=3, workers=4
   Validation: batch_size=2, workers=3
   Test: batch_size=1, workers=2

🔧 СОЗДАНИЕ ПРОСТЫХ DATALOADERS ДЛЯ CPU ТЕСТА:

📚 Создаем Training DataLoader...
   ✅ Training DataLoader: 3 batches
🔍 Создаем Validation DataLoader...
   ✅ Validation DataLoader: 3 batches
🧪 Создаем Test DataLoader...
   ✅ Test DataLoader: 5 batches

🧪 ТЕСТИРОВАНИЕ ЗАГРУЗКИ ДАННЫХ:
📚 Тест Training DataLoader...
   📊 Batch shape: torch.Size([3, 1, 128, 128, 64])
   🏷️ Labels shape: torch.Size([3])
   📈 Image range: [-0.200, 0.840]
   🏷️ Label values: tensor([0])
   ⏱️ Load time: 0.058s
   ✅ Training DataLoader test passed

🔍 Тест Validation DataLoader..

## 3.3: Data loading validation
Что мы делаем:
Тестируем integration реальных Dataset классов с DataLoaders и проверяем, что всё работает корректно перед переходом к model setup.

In [14]:
# ===================================
# ЯЧЕЙКА 3.3: DATA LOADING VALIDATION (GPU VERSION)
# ===================================

print("✅ ВАЛИДАЦИЯ ЗАГРУЗКИ РЕАЛЬНЫХ ДАННЫХ")
print("=" * 60)

# НА GPU: Используем полную версию CTAnomalyDataset из ячейки 3.1
# НА CPU: Мы уже протестировали упрощенную версию

print("📁 СОЗДАНИЕ PRODUCTION DATASETS С ПОЛНЫМИ ДАННЫМИ...")


# Training dataset (ВСЕ файлы)
print("📚 Production Training dataset...")
train_dataset = CTAnomalyDataset(
    normal_data_path=NORMAL_DATA_PATH,
    pathology_data_path=PATHOLOGY_DATA_PATH,
    file_list=train_files,  # ВСЕ 313 файлов
    split_type='train',
    ground_truth_labels=ground_truth_labels,
    cache_in_memory=False,  
    expected_shape=(128, 128, 64)
)
print(f"   ✅ Train dataset: {len(train_dataset)} файлов")

# Validation dataset (ВСЕ файлы)
print("🔍 Production Validation dataset...")
val_dataset = CTAnomalyDataset(
    normal_data_path=NORMAL_DATA_PATH,
    pathology_data_path=PATHOLOGY_DATA_PATH,
    file_list=val_files,  
    split_type='validation',
    ground_truth_labels=ground_truth_labels,
    cache_in_memory=True,  
    expected_shape=(128, 128, 64)
)
print(f"   ✅ Val dataset: {len(val_dataset)} файлов")

# Test dataset (ВСЕ файлы)
print("🧪 Production Test dataset...")
test_dataset = CTAnomalyDataset(
    normal_data_path=NORMAL_DATA_PATH,
    pathology_data_path=PATHOLOGY_DATA_PATH,
    file_list=test_files,  
    split_type='test',
    ground_truth_labels=ground_truth_labels,
    cache_in_memory=False,  
    expected_shape=(128, 128, 64)
)
print(f"   ✅ Test dataset: {len(test_dataset)} файлов")

print(f"\n📊 PRODUCTION DATASETS STATISTICS:")
for name, dataset in [('Train', train_dataset), ('Val', val_dataset), ('Test', test_dataset)]:
    info = dataset.get_split_info()
    print(f"   {name}: {info['total_files']} файлов ({info['normal_count']} норм, {info['pathology_count']} патол)")


✅ ВАЛИДАЦИЯ ЗАГРУЗКИ РЕАЛЬНЫХ ДАННЫХ
📁 СОЗДАНИЕ PRODUCTION DATASETS С ПОЛНЫМИ ДАННЫМИ...
📚 Production Training dataset...
✅ CTAnomalyDataset создан:
   Split: train
   Files: 313
   Normal files: 313
   Pathology files: 0
   Memory cache: Disabled
   ✅ Train dataset: 313 файлов
🔍 Production Validation dataset...
✅ CTAnomalyDataset создан:
   Split: validation
   Files: 229
   Normal files: 43
   Pathology files: 186
   Memory cache: Enabled
   ✅ Val dataset: 229 файлов
🧪 Production Test dataset...
✅ CTAnomalyDataset создан:
   Split: test
   Files: 249
   Normal files: 63
   Pathology files: 186
   Memory cache: Disabled
   ✅ Test dataset: 249 файлов

📊 PRODUCTION DATASETS STATISTICS:
   Train: 313 файлов (313 норм, 0 патол)
   Val: 229 файлов (43 норм, 186 патол)
   Test: 249 файлов (63 норм, 186 патол)


# ЭТАП 4: MODEL SETUP & CONFIGURATION

In [16]:
# ===================================
# ЯЧЕЙКА 4.2: PROGRESSIVE UNFREEZING TRAINING CONFIGURATION
# ===================================

print("🚀 PROGRESSIVE UNFREEZING TRAINING CONFIGURATION")
print("=" * 60)



# Training hyperparameters
TRAINING_CONFIG = {
    # Model
    'model': medvae_model,
    'device': device,
    
    # Training setup
    'num_epochs': 8,  # Увеличено для progressive strategy
    'strategy': 'progressive_unfreezing',
    'batch_size': 2,  # V100 оптимизированный
    
    # Progressive unfreezing schedule
    'progressive_schedule': {
        'decoder_only_epochs': 3,    # Было 2, стало 3
        'partial_encoder_epochs': 3, # Было 2, стало 3  
        'full_model_epochs': 2       # Было 2, стало 2
    },

    # Adaptive learning rates
    'learning_rates': {
        'decoder_only': 1e-4,     # Было 2e-4, уменьшили для 246M модели
        'partial_encoder': 5e-5,  # Было 1e-4, уменьшили
        'full_model': 2e-5        # Было 5e-5, уменьшили
    },

    
    # Loss function
    'reconstruction_loss': 'mse',
    'weight_decay': 1e-6,
    
    # Mixed precision
    'use_mixed_precision': True,
    'gradient_clipping': 1.0,
    
    # Monitoring
    'log_interval': 10,
    'save_interval': 2,
    'early_stopping_patience': 4,  # Больше patience для progressive
    
    # Checkpointing
    'save_best_model': True,
    'checkpoint_dir': 'checkpoints_progressive',
    'experiment_name': 'medvae_progressive_baseline'
}

print(f"📊 PROGRESSIVE TRAINING CONFIGURATION:")
for key, value in TRAINING_CONFIG.items():
    if key not in ['model']:
        print(f"   {key}: {value}")

# Создаем папку для checkpoints
import os
os.makedirs(TRAINING_CONFIG['checkpoint_dir'], exist_ok=True)
print(f"📁 Checkpoint directory: {TRAINING_CONFIG['checkpoint_dir']}")

# Progressive unfreezing function
def apply_progressive_strategy(model, epoch, config):
    """
    Применяет progressive unfreezing стратегию
    
    Args:
        model: MedVAE модель
        epoch: текущая эпоха (0-indexed)
        config: конфигурация training
    
    Returns:
        tuple: (current_lr, trainable_count, total_count)
    """
    
    decoder_epochs = config['progressive_schedule']['decoder_only_epochs']
    partial_epochs = config['progressive_schedule']['partial_encoder_epochs']
    
    print(f"\n🔄 PROGRESSIVE UNFREEZING - EPOCH {epoch+1}")
    
    # Определяем текущую фазу
    if epoch < decoder_epochs:
        # ФАЗА 1: Только decoder
        phase = "decoder_only"
        current_lr = config['learning_rates']['decoder_only']
        
        for name, param in model.named_parameters():
            # Ищем decoder-related параметры (может варьироваться в зависимости от архитектуры MedVAE)
            if any(keyword in name.lower() for keyword in ['decoder', 'dec', 'up', 'recon', 'output']):
                param.requires_grad = True
            else:
                param.requires_grad = False
        
        print("   🎯 ФАЗА 1: ❄️❄️❄️ → 🔥  (только decoder)")
        
    elif epoch < decoder_epochs + partial_epochs:
        # ФАЗА 2: Decoder + последние encoder layers
        phase = "partial_encoder"
        current_lr = config['learning_rates']['partial_encoder']
        
        for name, param in model.named_parameters():
            # Decoder всегда trainable
            if any(keyword in name.lower() for keyword in ['decoder', 'dec', 'up', 'recon', 'output']):
                param.requires_grad = True
            # Последние encoder layers (адаптируется под архитектуру)
            elif any(keyword in name.lower() for keyword in ['encoder.4', 'encoder.5', 'encoder.6', 'bottleneck', 'latent', 'enc.4', 'enc.5']):
                param.requires_grad = True
            # Batch norm и layer norm в последних слоях
            elif 'norm' in name.lower() and any(num in name for num in ['4', '5', '6']):
                param.requires_grad = True
            else:
                param.requires_grad = False
        
        print("   🎯 ФАЗА 2: ❄️❄️🔥 → 🔥  (+ последние encoder layers)")
        
    else:
        # ФАЗА 3: Все параметры
        phase = "full_model"
        current_lr = config['learning_rates']['full_model']
        
        for param in model.parameters():
            param.requires_grad = True
        
        print("   🎯 ФАЗА 3: 🔥🔥🔥 → 🔥  (все параметры)")
    
    # Подсчитываем параметры
    trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_count = sum(p.numel() for p in model.parameters())
    trainable_ratio = trainable_count / total_count * 100
    
    print(f"   📊 Trainable: {trainable_count:,}/{total_count:,} ({trainable_ratio:.1f}%)")
    print(f"   📈 Learning Rate: {current_lr:.2e}")
    
    return current_lr, trainable_count, total_count, phase

# Loss function setup
criterion = nn.MSELoss()
print(f"🎯 Loss function: MSE")

# Mixed precision scaler
if TRAINING_CONFIG['use_mixed_precision'] and device.type == 'cuda':
    scaler = GradScaler()
    print(f"⚡ Mixed precision: Enabled")
else:
    scaler = None
    print(f"⚡ Mixed precision: Disabled")

# Начальная настройка модели (фаза 1)
initial_lr, initial_trainable, total_params, initial_phase = apply_progressive_strategy(
    medvae_model, 0, TRAINING_CONFIG
)

# Создаем начальный optimizer
trainable_params = [p for p in medvae_model.parameters() if p.requires_grad]
optimizer = optim.AdamW(
    trainable_params,
    lr=initial_lr,
    weight_decay=TRAINING_CONFIG['weight_decay'],
    betas=(0.9, 0.999),
    eps=1e-8
)

print(f"⚙️ Initial optimizer created for {len(trainable_params):,} parameters")

# Scheduler (будет пересоздаваться для каждой фазы)
scheduler = None  # Создадим в training loop для каждой фазы

# DataLoaders с GPU конфигурацией
print(f"\n📊 СОЗДАНИЕ PRODUCTION DATALOADERS...")

train_dataset = CTAnomalyDataset(
    normal_data_path=NORMAL_DATA_PATH,
    pathology_data_path=PATHOLOGY_DATA_PATH,
    file_list=train_files,
    split_type='train',
    ground_truth_labels=ground_truth_labels,
    cache_in_memory=False,
    expected_shape=(128, 128, 64)
)

val_dataset = CTAnomalyDataset(
    normal_data_path=NORMAL_DATA_PATH,
    pathology_data_path=PATHOLOGY_DATA_PATH,
    file_list=val_files,
    split_type='validation',
    ground_truth_labels=ground_truth_labels,
    cache_in_memory=True,  # 32GB RAM позволяет
    expected_shape=(128, 128, 64)
)

test_dataset = CTAnomalyDataset(
    normal_data_path=NORMAL_DATA_PATH,
    pathology_data_path=PATHOLOGY_DATA_PATH,
    file_list=test_files,
    split_type='test',
    ground_truth_labels=ground_truth_labels,
    cache_in_memory=False,
    expected_shape=(128, 128, 64)
)

# GPU DataLoaders
train_loader = DataLoader(train_dataset, **dataloader_configs['gpu_train_config'])
val_loader = DataLoader(val_dataset, **dataloader_configs['gpu_val_config'])
test_loader = DataLoader(test_dataset, **dataloader_configs['gpu_test_config'])

print(f"✅ DataLoaders созданы:")
print(f"   Train: {len(train_loader)} batches ({len(train_dataset)} samples)")
print(f"   Validation: {len(val_loader)} batches ({len(val_dataset)} samples)")
print(f"   Test: {len(test_loader)} batches ({len(test_dataset)} samples)")

# Функция для обновления optimizer при смене фазы
def update_optimizer_for_phase(model, phase_lr, config):
    """Создает новый optimizer для текущей фазы"""
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    
    new_optimizer = optim.AdamW(
        trainable_params,
        lr=phase_lr,
        weight_decay=config['weight_decay'],
        betas=(0.9, 0.999),
        eps=1e-8
    )
    
    return new_optimizer

print(f"\n🚀 PROGRESSIVE UNFREEZING CONFIGURATION ГОТОВА")
print(f"   📊 Strategy: 3-phase progressive unfreezing")
print(f"   ⏱️ Schedule: {TRAINING_CONFIG['progressive_schedule']}")
print(f"   📈 LR Schedule: {TRAINING_CONFIG['learning_rates']}")
print(f"   🎯 Target AUC: 0.86-0.92")

# Сохраняем конфигурацию для использования в training loop
progressive_config = TRAINING_CONFIG.copy()
print(f"💾 Progressive config готов к training loop")


🚀 PROGRESSIVE UNFREEZING TRAINING CONFIGURATION
📊 PROGRESSIVE TRAINING CONFIGURATION:
   device: cpu
   num_epochs: 8
   strategy: progressive_unfreezing
   batch_size: 2
   progressive_schedule: {'decoder_only_epochs': 3, 'partial_encoder_epochs': 3, 'full_model_epochs': 2}
   learning_rates: {'decoder_only': 0.0001, 'partial_encoder': 5e-05, 'full_model': 2e-05}
   reconstruction_loss: mse
   weight_decay: 1e-06
   use_mixed_precision: True
   gradient_clipping: 1.0
   log_interval: 10
   save_interval: 2
   early_stopping_patience: 4
   save_best_model: True
   checkpoint_dir: checkpoints_progressive
   experiment_name: medvae_progressive_baseline
📁 Checkpoint directory: checkpoints_progressive
🎯 Loss function: MSE
⚡ Mixed precision: Disabled

🔄 PROGRESSIVE UNFREEZING - EPOCH 1
   🎯 ФАЗА 1: ❄️❄️❄️ → 🔥  (только decoder)
   📊 Trainable: 145,963,265/246,077,309 (59.3%)
   📈 Learning Rate: 1.00e-04
⚙️ Initial optimizer created for 138 parameters

📊 СОЗДАНИЕ PRODUCTION DATALOADERS...
✅ C

In [17]:
# ===================================
# ЯЧЕЙКА 4.3: TRAINING MONITORING SETUP
# ===================================

print("📊 НАСТРОЙКА МОНИТОРИНГА ОБУЧЕНИЯ")
print("=" * 60)

# Metrics tracking
training_metrics = {
    'train_loss': [],
    'val_loss': [],
    'learning_rates': [],
    'epoch_times': [],
    'val_auc': [],
    'best_val_loss': float('inf'),
    'best_epoch': 0,
    'total_training_time': 0
}

def compute_anomaly_scores(model, dataloader, device):
    """Вычисляет anomaly scores для AUC мониторинга"""
    model.eval()
    reconstruction_errors = []
    labels = []
    
    with torch.no_grad():
        for batch in dataloader:
            if isinstance(batch, dict):
                images = batch['image'].to(device)
                batch_labels = batch['label'].numpy() if isinstance(batch['label'], torch.Tensor) else batch['label']
            else:
                images, batch_labels = batch
                images = images.to(device)
                batch_labels = batch_labels.numpy() if isinstance(batch_labels, torch.Tensor) else batch_labels
            
            # Reconstruction
            if scaler and TRAINING_CONFIG['use_mixed_precision']:
                with autocast():
                    reconstructed = model(images)
            else:
                reconstructed = model(images)
            
            # MSE для каждого образца
            mse_errors = torch.mean((images - reconstructed) ** 2, dim=[1, 2, 3, 4])
            reconstruction_errors.extend(mse_errors.cpu().numpy())
            labels.extend(batch_labels)
    
    return np.array(reconstruction_errors), np.array(labels)

def log_metrics(epoch, train_loss, val_loss, val_auc, lr, epoch_time):
    """Логирование метрик"""
    training_metrics['train_loss'].append(train_loss)
    training_metrics['val_loss'].append(val_loss)
    training_metrics['val_auc'].append(val_auc)
    training_metrics['learning_rates'].append(lr)
    training_metrics['epoch_times'].append(epoch_time)
    
    print(f"📊 Epoch {epoch+1}:")
    print(f"   Train Loss: {train_loss:.6f}")
    print(f"   Val Loss: {val_loss:.6f}")
    print(f"   Val AUC: {val_auc:.4f}")
    print(f"   Learning Rate: {lr:.2e}")
    print(f"   Epoch Time: {epoch_time:.1f}s")
    
    # Сохраняем лучшую модель
    if val_loss < training_metrics['best_val_loss']:
        training_metrics['best_val_loss'] = val_loss
        training_metrics['best_epoch'] = epoch + 1
        
        if TRAINING_CONFIG['save_best_model']:
            best_model_path = f"{TRAINING_CONFIG['checkpoint_dir']}/best_model.pth"
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
                'loss': val_loss,
                'auc': val_auc,
                'training_config': TRAINING_CONFIG
            }, best_model_path)
            print(f"   💾 Best model saved: AUC {val_auc:.4f}")

def plot_training_progress():
    """Визуализация процесса обучения"""
    if len(training_metrics['train_loss']) < 2:
        return
        
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    epochs = list(range(1, len(training_metrics['train_loss']) + 1))
    
    # Loss curves
    axes[0,0].plot(epochs, training_metrics['train_loss'], 'b-', label='Train Loss')
    axes[0,0].plot(epochs, training_metrics['val_loss'], 'r-', label='Val Loss')
    axes[0,0].set_title('Training & Validation Loss')
    axes[0,0].set_xlabel('Epoch')
    axes[0,0].set_ylabel('Loss')
    axes[0,0].legend()
    axes[0,0].grid(True)
    
    # AUC curve
    axes[0,1].plot(epochs, training_metrics['val_auc'], 'g-', label='Val AUC')
    axes[0,1].set_title('Validation AUC')
    axes[0,1].set_xlabel('Epoch')
    axes[0,1].set_ylabel('AUC')
    axes[0,1].legend()
    axes[0,1].grid(True)
    
    # Learning rate
    axes[1,0].plot(epochs, training_metrics['learning_rates'], 'purple', label='Learning Rate')
    axes[1,0].set_title('Learning Rate Schedule')
    axes[1,0].set_xlabel('Epoch')
    axes[1,0].set_ylabel('LR')
    axes[1,0].set_yscale('log')
    axes[1,0].legend()
    axes[1,0].grid(True)
    
    # Epoch times
    axes[1,1].bar(epochs, training_metrics['epoch_times'], alpha=0.7)
    axes[1,1].set_title('Epoch Training Time')
    axes[1,1].set_xlabel('Epoch')
    axes[1,1].set_ylabel('Time (s)')
    axes[1,1].grid(True)
    
    plt.tight_layout()
    plt.savefig(f"{TRAINING_CONFIG['checkpoint_dir']}/training_progress.png", dpi=150, bbox_inches='tight')
    plt.show()

def save_training_log():
    """Сохранение лога обучения"""
    log_path = f"{TRAINING_CONFIG['checkpoint_dir']}/training_log.json"
    
    log_data = {
        'training_config': TRAINING_CONFIG.copy(),
        'metrics': training_metrics.copy(),
        'dataset_info': {
            'train_size': len(train_dataset),
            'val_size': len(val_dataset),
            'test_size': len(test_dataset)
        }
    }
    
    # Убираем model из config для JSON сериализации
    if 'model' in log_data['training_config']:
        del log_data['training_config']['model']
    
    with open(log_path, 'w') as f:
        json.dump(log_data, f, indent=2)
    
    print(f"📄 Training log saved: {log_path}")

print("✅ Monitoring setup готов")
print("   📊 Metrics tracking инициализирован")
print("   🎨 Plotting functions готовы")
print("   💾 Logging система настроена")

📊 НАСТРОЙКА МОНИТОРИНГА ОБУЧЕНИЯ
✅ Monitoring setup готов
   📊 Metrics tracking инициализирован
   🎨 Plotting functions готовы
   💾 Logging система настроена


# ЭТАП 5: TRAINING EXECUTION

In [ ]:
# ===================================
# ЯЧЕЙКА 5.1: PROGRESSIVE TRAINING LOOP
# ===================================

print("🚀 ЗАПУСК PROGRESSIVE TRAINING LOOP")
print("=" * 60)

import time
from tqdm import tqdm

def train_epoch_progressive(model, train_loader, optimizer, criterion, device, scaler=None, epoch=0):
    """Одна эпоха обучения с progressive unfreezing"""
    model.train()
    total_loss = 0
    num_batches = 0
    
    progress_bar = tqdm(train_loader, desc=f"Training Epoch {epoch+1}", leave=False)
    
    for batch_idx, batch in enumerate(progress_bar):
        # Извлекаем данные
        if isinstance(batch, dict):
            images = batch['image'].to(device, non_blocking=True)
        else:
            images, _ = batch  # TensorDataset fallback
            images = images.to(device, non_blocking=True)
        
        # Skip пустых батчей
        if images.size(0) == 0:
            continue
            
        optimizer.zero_grad()
        
        # Forward pass с mixed precision
        if scaler and progressive_config['use_mixed_precision']:
            with autocast():
                reconstructed = model(images)
                loss = criterion(reconstructed, images)
        else:
            reconstructed = model(images)
            loss = criterion(reconstructed, images)
        
        # Backward pass
        if scaler and progressive_config['use_mixed_precision']:
            scaler.scale(loss).backward()
            if progressive_config.get('gradient_clipping'):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), progressive_config['gradient_clipping'])
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            if progressive_config.get('gradient_clipping'):
                torch.nn.utils.clip_grad_norm_(model.parameters(), progressive_config['gradient_clipping'])
            optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
        
        # Update progress bar
        current_phase = "Dec" if epoch < 2 else "Dec+Enc" if epoch < 4 else "Full"
        progress_bar.set_postfix({
            'phase': current_phase,
            'loss': f'{loss.item():.6f}',
            'avg_loss': f'{total_loss/num_batches:.6f}',
            'lr': f'{optimizer.param_groups[0]["lr"]:.2e}'
        })
        
        # Периодическая очистка кэша
        if batch_idx % 50 == 0:
            torch.cuda.empty_cache()
    
    return total_loss / num_batches if num_batches > 0 else 0

def validate_epoch_progressive(model, val_loader, criterion, device, epoch=0):
    """Валидация с AUC вычислением для progressive training"""
    model.eval()
    total_loss = 0
    num_batches = 0
    
    # Собираем данные для AUC
    reconstruction_errors = []
    labels = []
    
    current_phase = "Dec" if epoch < 2 else "Dec+Enc" if epoch < 4 else "Full"
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Validation {current_phase}", leave=False):
            # Извлекаем данные
            if isinstance(batch, dict):
                images = batch['image'].to(device, non_blocking=True)
                batch_labels = batch['label']
            else:
                images, batch_labels = batch
                images = images.to(device, non_blocking=True)
            
            # Skip пустых батчей
            if images.size(0) == 0:
                continue
            
            # Forward pass
            if scaler and progressive_config['use_mixed_precision']:
                with autocast():
                    reconstructed = model(images)
                    loss = criterion(reconstructed, images)
            else:
                reconstructed = model(images)
                loss = criterion(reconstructed, images)
            
            total_loss += loss.item()
            num_batches += 1
            
            # Вычисляем reconstruction errors для AUC
            mse_errors = torch.mean((images - reconstructed) ** 2, dim=[1, 2, 3, 4])
            reconstruction_errors.extend(mse_errors.cpu().numpy())
            
            # Labels
            if isinstance(batch_labels, torch.Tensor):
                labels.extend(batch_labels.numpy())
            else:
                labels.extend(batch_labels)
    
    avg_loss = total_loss / num_batches if num_batches > 0 else float('inf')
    
    # Вычисляем AUC
    if len(reconstruction_errors) > 0 and len(set(labels)) > 1:
        try:
            auc = roc_auc_score(labels, reconstruction_errors)
        except Exception as e:
            print(f"⚠️ AUC calculation failed: {e}")
            auc = 0.5
    else:
        auc = 0.5
    
    return avg_loss, auc

# Расширенные training metrics для progressive tracking
progressive_training_metrics = {
    'train_loss': [],
    'val_loss': [],
    'learning_rates': [],
    'epoch_times': [],
    'val_auc': [],
    'training_phases': [],  # Новое: отслеживание фаз
    'trainable_params': [],  # Новое: количество trainable параметров
    'best_val_loss': float('inf'),
    'best_epoch': 0,
    'total_training_time': 0,
    'phase_transitions': []  # Новое: моменты смены фаз
}

def log_progressive_metrics(epoch, train_loss, val_loss, val_auc, lr, epoch_time, phase, trainable_count):
    """Расширенное логирование метрик для progressive training"""
    progressive_training_metrics['train_loss'].append(train_loss)
    progressive_training_metrics['val_loss'].append(val_loss)
    progressive_training_metrics['val_auc'].append(val_auc)
    progressive_training_metrics['learning_rates'].append(lr)
    progressive_training_metrics['epoch_times'].append(epoch_time)
    progressive_training_metrics['training_phases'].append(phase)
    progressive_training_metrics['trainable_params'].append(trainable_count)
    
    print(f"📊 Epoch {epoch+1} ({phase.upper()}):")
    print(f"   Train Loss: {train_loss:.6f}")
    print(f"   Val Loss: {val_loss:.6f}")
    print(f"   Val AUC: {val_auc:.4f}")
    print(f"   Learning Rate: {lr:.2e}")
    print(f"   Trainable Params: {trainable_count:,}")
    print(f"   Epoch Time: {epoch_time:.1f}s")
    
    # Отмечаем transitions между фазами
    if len(progressive_training_metrics['training_phases']) > 1:
        prev_phase = progressive_training_metrics['training_phases'][-2]
        if phase != prev_phase:
            progressive_training_metrics['phase_transitions'].append(epoch + 1)
            print(f"   🔄 PHASE TRANSITION: {prev_phase} → {phase}")
    
    # Сохраняем лучшую модель
    if val_loss < progressive_training_metrics['best_val_loss']:
        progressive_training_metrics['best_val_loss'] = val_loss
        progressive_training_metrics['best_epoch'] = epoch + 1
        
        if progressive_config['save_best_model']:
            best_model_path = f"{progressive_config['checkpoint_dir']}/best_model_progressive.pth"
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': medvae_model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': val_loss,
                'auc': val_auc,
                'phase': phase,
                'trainable_count': trainable_count,
                'training_config': progressive_config,
                'progressive_metrics': progressive_training_metrics.copy()
            }, best_model_path)
            print(f"   💾 Best model saved: AUC {val_auc:.4f} (Phase: {phase})")

# Основной progressive training loop
print(f"🏃‍♂️ Начинаем progressive обучение на {progressive_config['num_epochs']} эпох")
print(f"📊 Dataset sizes: Train={len(train_dataset)}, Val={len(val_dataset)}")
print(f"🎯 Strategy: {progressive_config['progressive_schedule']}")

total_start_time = time.time()
current_phase = ""
current_optimizer = optimizer  # Из предыдущей ячейки

try:
    for epoch in range(progressive_config['num_epochs']):
        print(f"\n🔄 EPOCH {epoch+1}/{progressive_config['num_epochs']}")
        epoch_start_time = time.time()
        
        # Применяем progressive unfreezing strategy
        phase_lr, trainable_count, total_params, phase = apply_progressive_strategy(
            medvae_model, epoch, progressive_config
        )
        
        # Обновляем optimizer если сменилась фаза
        if phase != current_phase:
            print(f"🔄 Updating optimizer for phase: {phase}")
            current_optimizer = update_optimizer_for_phase(medvae_model, phase_lr, progressive_config)
            current_phase = phase
        
        # Training
        train_loss = train_epoch_progressive(
            medvae_model, train_loader, current_optimizer, criterion, device, scaler, epoch
        )
        
        # Validation
        val_loss, val_auc = validate_epoch_progressive(medvae_model, val_loader, criterion, device, epoch)
        
        # Epoch timing
        epoch_time = time.time() - epoch_start_time
        current_lr = current_optimizer.param_groups[0]['lr']
        
        # Log progressive metrics
        log_progressive_metrics(epoch, train_loss, val_loss, val_auc, current_lr, epoch_time, phase, trainable_count)
        
        # Сохраняем checkpoint каждые save_interval эпох
        if (epoch + 1) % progressive_config['save_interval'] == 0:
            checkpoint_path = f"{progressive_config['checkpoint_dir']}/checkpoint_progressive_epoch_{epoch+1}.pth"
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': medvae_model.state_dict(),
                'optimizer_state_dict': current_optimizer.state_dict(),
                'loss': val_loss,
                'auc': val_auc,
                'phase': phase,
                'trainable_count': trainable_count,
                'progressive_metrics': progressive_training_metrics.copy(),
                'training_config': progressive_config
            }, checkpoint_path)
            print(f"   💾 Checkpoint saved: {checkpoint_path}")
        
        # Early stopping check (более длительный для progressive)
        if len(progressive_training_metrics['val_loss']) >= progressive_config['early_stopping_patience']:
            recent_losses = progressive_training_metrics['val_loss'][-progressive_config['early_stopping_patience']:]
            if all(recent_losses[i] <= recent_losses[i+1] for i in range(len(recent_losses)-1)):
                print(f"🛑 Early stopping triggered after {epoch+1} epochs")
                break
        
        # Progressive plotting каждые 2 эпохи
        if (epoch + 1) % 2 == 0:
            plot_training_progress()
        
        # Memory cleanup
        torch.cuda.empty_cache()
    
    # Final results
    total_training_time = time.time() - total_start_time
    progressive_training_metrics['total_training_time'] = total_training_time
    
    print(f"\n🎉 PROGRESSIVE TRAINING ЗАВЕРШЕН!")
    print(f"   ⏱️ Total time: {total_training_time/60:.1f} minutes")
    print(f"   🏆 Best epoch: {progressive_training_metrics['best_epoch']}")
    print(f"   📊 Best val loss: {progressive_training_metrics['best_val_loss']:.6f}")
    print(f"   🎯 Final val AUC: {progressive_training_metrics['val_auc'][-1]:.4f}")
    print(f"   🔄 Phase transitions: {progressive_training_metrics['phase_transitions']}")
    
    # Final plot и save
    plot_training_progress()
    save_training_log()
    
    print(f"\n✅ Progressive training успешно завершен")
    
except KeyboardInterrupt:
    print(f"\n⚠️ Progressive обучение прервано пользователем")
    save_training_log()
    
except Exception as e:
    print(f"\n❌ Ошибка во время progressive обучения: {e}")
    import traceback
    traceback.print_exc()
    save_training_log()

# Функции для progressive visualization и logging
def plot_progressive_training_progress():
    """Визуализация progressive training с фазами"""
    if len(progressive_training_metrics['train_loss']) < 2:
        return
        
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    epochs = list(range(1, len(progressive_training_metrics['train_loss']) + 1))
    
    # Loss curves с фазовыми разделителями
    axes[0,0].plot(epochs, progressive_training_metrics['train_loss'], 'b-', label='Train Loss')
    axes[0,0].plot(epochs, progressive_training_metrics['val_loss'], 'r-', label='Val Loss')
    
    # Добавляем вертикальные линии для phase transitions
    for transition_epoch in progressive_training_metrics['phase_transitions']:
        axes[0,0].axvline(x=transition_epoch, color='gray', linestyle='--', alpha=0.7)
    
    axes[0,0].set_title('Progressive Training & Validation Loss')
    axes[0,0].set_xlabel('Epoch')
    axes[0,0].set_ylabel('Loss')
    axes[0,0].legend()
    axes[0,0].grid(True)
    
    # AUC curve
    axes[0,1].plot(epochs, progressive_training_metrics['val_auc'], 'g-', label='Val AUC')
    for transition_epoch in progressive_training_metrics['phase_transitions']:
        axes[0,1].axvline(x=transition_epoch, color='gray', linestyle='--', alpha=0.7)
    axes[0,1].set_title('Validation AUC Progress')
    axes[0,1].set_xlabel('Epoch')
    axes[0,1].set_ylabel('AUC')
    axes[0,1].legend()
    axes[0,1].grid(True)
    
    # Learning rate
    axes[0,2].plot(epochs, progressive_training_metrics['learning_rates'], 'purple', label='Learning Rate')
    for transition_epoch in progressive_training_metrics['phase_transitions']:
        axes[0,2].axvline(x=transition_epoch, color='gray', linestyle='--', alpha=0.7)
    axes[0,2].set_title('Progressive Learning Rate Schedule')
    axes[0,2].set_xlabel('Epoch')
    axes[0,2].set_ylabel('LR')
    axes[0,2].set_yscale('log')
    axes[0,2].legend()
    axes[0,2].grid(True)
    
    # Trainable parameters over time
    axes[1,0].plot(epochs, progressive_training_metrics['trainable_params'], 'orange', label='Trainable Params')
    for transition_epoch in progressive_training_metrics['phase_transitions']:
        axes[1,0].axvline(x=transition_epoch, color='gray', linestyle='--', alpha=0.7)
    axes[1,0].set_title('Trainable Parameters by Phase')
    axes[1,0].set_xlabel('Epoch')
    axes[1,0].set_ylabel('Parameter Count')
    axes[1,0].legend()
    axes[1,0].grid(True)
    
    # Epoch times
    axes[1,1].bar(epochs, progressive_training_metrics['epoch_times'], alpha=0.7)
    axes[1,1].set_title('Epoch Training Time')
    axes[1,1].set_xlabel('Epoch')
    axes[1,1].set_ylabel('Time (s)')
    axes[1,1].grid(True)
    
    # Phase timeline
    axes[1,2].scatter(epochs, [0]*len(epochs), c=range(len(epochs)), cmap='viridis', s=50)
    phase_colors = {'decoder_only': 'blue', 'partial_encoder': 'orange', 'full_model': 'red'}
    for i, (epoch, phase) in enumerate(zip(epochs, progressive_training_metrics['training_phases'])):
        axes[1,2].scatter(epoch, 0, c=phase_colors.get(phase, 'gray'), s=100, alpha=0.7)
    axes[1,2].set_title('Training Phase Timeline')
    axes[1,2].set_xlabel('Epoch')
    axes[1,2].set_yticks([])
    
    plt.tight_layout()
    plt.savefig(f"{progressive_config['checkpoint_dir']}/progressive_training_progress.png", dpi=150, bbox_inches='tight')
    plt.show()

def save_progressive_training_log():
    """Сохранение лога progressive обучения"""
    log_path = f"{progressive_config['checkpoint_dir']}/progressive_training_log.json"
    
    log_data = {
        'training_config': progressive_config.copy(),
        'progressive_metrics': progressive_training_metrics.copy(),
        'dataset_info': {
            'train_size': len(train_dataset),
            'val_size': len(val_dataset),
            'test_size': len(test_dataset)
        }
    }
    
    # Убираем model из config для JSON сериализации
    if 'model' in log_data['training_config']:
        del log_data['training_config']['model']
    
    with open(log_path, 'w') as f:
        json.dump(log_data, f, indent=2)
    
    print(f"📄 Progressive training log saved: {log_path}")

print(f"\n🚀 Готовы к финальному progressive evaluation")

# Обновляем переменную для использования в следующих ячейках
training_metrics = progressive_training_metrics

# ЭТАП 6: FINAL EVALUATION & RESULTS

In [ ]:
# ===================================
# ЯЧЕЙКА 6.1: FINAL EVALUATION & RESULTS
# ===================================

print("🎯 ФИНАЛЬНОЕ EVALUATION НА TEST SET")
print("=" * 60)

# Загружаем лучшую модель
print("📥 Загружаем лучшую модель...")
best_model_path = f"{TRAINING_CONFIG['checkpoint_dir']}/best_model.pth"

if os.path.exists(best_model_path):
    checkpoint = torch.load(best_model_path, map_location=device)
    medvae_model.load_state_dict(checkpoint['model_state_dict'])
    best_epoch = checkpoint['epoch']
    best_val_loss = checkpoint['loss']
    print(f"✅ Best model loaded (Epoch {best_epoch}, Val Loss: {best_val_loss:.6f})")
else:
    print("⚠️ Best model не найден, используем текущую модель")

# Функция для полного evaluation
def comprehensive_evaluation(model, test_loader, device):
    """Полное evaluation с метриками"""
    model.eval()
    
    all_reconstruction_errors = []
    all_labels = []
    all_filenames = []
    
    print("🔄 Обрабатываем test set...")
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Test evaluation"):
            # Извлекаем данные
            if isinstance(batch, dict):
                images = batch['image'].to(device, non_blocking=True)
                labels = batch['label']
                filenames = batch['filename']
            else:
                images, labels = batch
                images = images.to(device, non_blocking=True)
                filenames = [f"sample_{i}" for i in range(len(images))]
            
            if images.size(0) == 0:
                continue
            
            # Forward pass
            if scaler and TRAINING_CONFIG['use_mixed_precision']:
                with autocast():
                    reconstructed = model(images)
            else:
                reconstructed = model(images)
            
            # Reconstruction errors
            mse_errors = torch.mean((images - reconstructed) ** 2, dim=[1, 2, 3, 4])
            
            # Collect results
            all_reconstruction_errors.extend(mse_errors.cpu().numpy())
            
            if isinstance(labels, torch.Tensor):
                all_labels.extend(labels.numpy())
            else:
                all_labels.extend(labels)
            
            if isinstance(filenames, list):
                all_filenames.extend(filenames)
            else:
                all_filenames.extend([filenames])
    
    return np.array(all_reconstruction_errors), np.array(all_labels), all_filenames

# Выполняем evaluation
test_errors, test_labels, test_filenames = comprehensive_evaluation(medvae_model, test_loader, device)

print(f"📊 Test set evaluation:")
print(f"   Samples processed: {len(test_errors)}")
print(f"   Normal samples: {np.sum(test_labels == 0)}")
print(f"   Pathology samples: {np.sum(test_labels == 1)}")

# Вычисляем статистику нормальных данных из validation
print(f"\n📊 Вычисляем baseline статистику...")

# Используем нормальные образцы из validation для threshold
val_errors, val_labels, _ = comprehensive_evaluation(medvae_model, val_loader, device)
normal_val_errors = val_errors[val_labels == 0]

if len(normal_val_errors) > 0:
    normal_mean = np.mean(normal_val_errors)
    normal_std = np.std(normal_val_errors)
    threshold_2sigma = normal_mean + 2 * normal_std
    threshold_3sigma = normal_mean + 3 * normal_std
    
    print(f"   Normal data statistics:")
    print(f"      Mean error: {normal_mean:.6f}")
    print(f"      Std error: {normal_std:.6f}")
    print(f"      Threshold (μ + 2σ): {threshold_2sigma:.6f}")
    print(f"      Threshold (μ + 3σ): {threshold_3sigma:.6f}")
else:
    print("⚠️ Нет нормальных данных в validation для статистики")
    threshold_2sigma = np.median(test_errors)
    threshold_3sigma = np.percentile(test_errors, 95)

# ROC AUC
if len(np.unique(test_labels)) > 1:
    auc_score = roc_auc_score(test_labels, test_errors)
    print(f"\n🎯 ROC AUC: {auc_score:.4f}")
    
    # ROC Curve
    fpr, tpr, roc_thresholds = roc_curve(test_labels, test_errors)
    
    # Precision-Recall Curve
    precision, recall, pr_thresholds = precision_recall_curve(test_labels, test_errors)
    
    # Optimal threshold (maximize F1-score)
    f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
    best_f1_idx = np.argmax(f1_scores)
    optimal_threshold = pr_thresholds[best_f1_idx] if best_f1_idx < len(pr_thresholds) else threshold_2sigma
    best_f1 = f1_scores[best_f1_idx]
    
    print(f"🏆 Best F1-score: {best_f1:.4f}")
    print(f"🎯 Optimal threshold: {optimal_threshold:.6f}")
    
    # Classification report с optimal threshold
    y_pred = (test_errors > optimal_threshold).astype(int)
    
    print(f"\n📋 CLASSIFICATION REPORT (Optimal Threshold):")
    print(classification_report(test_labels, y_pred, target_names=['Normal', 'Pathology']))
    
    # Confusion Matrix
    cm = confusion_matrix(test_labels, y_pred)
    print(f"\n📊 CONFUSION MATRIX:")
    print(f"         Predicted")
    print(f"         Normal  Pathology")
    print(f"Actual Normal    {cm[0,0]:6d}     {cm[0,1]:6d}")
    print(f"       Pathology {cm[1,0]:6d}     {cm[1,1]:6d}")
    
    # Дополнительные метрики
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0
    
    print(f"\n📈 ДОПОЛНИТЕЛЬНЫЕ МЕТРИКИ:")
    print(f"   Sensitivity (Recall): {sensitivity:.4f}")
    print(f"   Specificity: {specificity:.4f}")
    print(f"   PPV (Precision): {ppv:.4f}")
    print(f"   NPV: {npv:.4f}")
    
else:
    print("⚠️ Недостаточно классов для вычисления AUC")
    auc_score = 0.5
    optimal_threshold = threshold_2sigma

print(f"\n✅ Final evaluation завершен")


In [ ]:
# ===================================
# ЯЧЕЙКА 6.2: VISUALIZATIONS AND ANALYSIS
# ===================================

print("🎨 СОЗДАНИЕ ВИЗУАЛИЗАЦИЙ И АНАЛИЗА")
print("=" * 60)

# Создаем comprehensive visualization
fig = plt.figure(figsize=(20, 15))

# ROC Curve
plt.subplot(3, 3, 1)
if len(np.unique(test_labels)) > 1:
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {auc_score:.3f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend(loc="lower right")
    plt.grid(True)

# Precision-Recall Curve
plt.subplot(3, 3, 2)
if len(np.unique(test_labels)) > 1:
    plt.plot(recall, precision, color='blue', lw=2, label=f'PR (Best F1 = {best_f1:.3f})')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curve')
    plt.legend()
    plt.grid(True)

# Score Distribution
plt.subplot(3, 3, 3)
normal_errors = test_errors[test_labels == 0]
pathology_errors = test_errors[test_labels == 1]

if len(normal_errors) > 0:
    plt.hist(normal_errors, bins=30, alpha=0.7, label=f'Normal (n={len(normal_errors)})', color='green')
if len(pathology_errors) > 0:
    plt.hist(pathology_errors, bins=30, alpha=0.7, label=f'Pathology (n={len(pathology_errors)})', color='red')

plt.axvline(optimal_threshold, color='black', linestyle='--', label=f'Threshold = {optimal_threshold:.4f}')
plt.xlabel('Reconstruction Error')
plt.ylabel('Frequency')
plt.title('Error Distribution')
plt.legend()
plt.grid(True)

# Confusion Matrix Heatmap
plt.subplot(3, 3, 4)
if len(np.unique(test_labels)) > 1:
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Normal', 'Pathology'],
                yticklabels=['Normal', 'Pathology'])
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')

# Training Loss Curves
plt.subplot(3, 3, 5)
if len(training_metrics['train_loss']) > 0:
    epochs = range(1, len(training_metrics['train_loss']) + 1)
    plt.plot(epochs, training_metrics['train_loss'], 'b-', label='Train Loss')
    plt.plot(epochs, training_metrics['val_loss'], 'r-', label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training Progress')
    plt.legend()
    plt.grid(True)

# Validation AUC Progress
plt.subplot(3, 3, 6)
if len(training_metrics['val_auc']) > 0:
    epochs = range(1, len(training_metrics['val_auc']) + 1)
    plt.plot(epochs, training_metrics['val_auc'], 'g-', label='Val AUC')
    plt.xlabel('Epoch')
    plt.ylabel('AUC')
    plt.title('Validation AUC Progress')
    plt.legend()
    plt.grid(True)

# Box plot of errors by class
plt.subplot(3, 3, 7)
if len(normal_errors) > 0 and len(pathology_errors) > 0:
    data_to_plot = [normal_errors, pathology_errors]
    plt.boxplot(data_to_plot, labels=['Normal', 'Pathology'])
    plt.ylabel('Reconstruction Error')
    plt.title('Error Distribution by Class')
    plt.grid(True)

# Learning Rate Schedule
plt.subplot(3, 3, 8)
if len(training_metrics['learning_rates']) > 0:
    epochs = range(1, len(training_metrics['learning_rates']) + 1)
    plt.plot(epochs, training_metrics['learning_rates'], 'purple', label='Learning Rate')
    plt.xlabel('Epoch')
    plt.ylabel('Learning Rate')
    plt.title('Learning Rate Schedule')
    plt.yscale('log')
    plt.legend()
    plt.grid(True)

# Threshold Analysis
plt.subplot(3, 3, 9)
thresholds = np.linspace(test_errors.min(), test_errors.max(), 100)
f1_scores_thresh = []

for thresh in thresholds:
    y_pred_thresh = (test_errors > thresh).astype(int)
    if len(np.unique(y_pred_thresh)) > 1 and len(np.unique(test_labels)) > 1:
        from sklearn.metrics import f1_score
        f1 = f1_score(test_labels, y_pred_thresh)
        f1_scores_thresh.append(f1)
    else:
        f1_scores_thresh.append(0)

plt.plot(thresholds, f1_scores_thresh, 'orange', label='F1-Score')
plt.axvline(optimal_threshold, color='red', linestyle='--', label=f'Optimal = {optimal_threshold:.4f}')
plt.xlabel('Threshold')
plt.ylabel('F1-Score')
plt.title('Threshold vs F1-Score')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig(f"{TRAINING_CONFIG['checkpoint_dir']}/final_analysis.png", dpi=300, bbox_inches='tight')
plt.show()

print("✅ Визуализации созданы и сохранены")

In [ ]:
# ===================================
# ЯЧЕЙКА 6.3: ERROR ANALYSIS
# ===================================

print("🔍 ДЕТАЛЬНЫЙ АНАЛИЗ ОШИБОК")
print("=" * 60)

# Анализ самых сложных случаев
def analyze_difficult_cases(errors, labels, filenames, top_k=5):
    """Анализ наиболее сложных для классификации случаев"""
    
    # False Positives (Normal classified as Pathology)
    normal_mask = labels == 0
    normal_errors = errors[normal_mask]
    normal_files = np.array(filenames)[normal_mask]
    
    if len(normal_errors) > 0:
        fp_indices = np.argsort(normal_errors)[-top_k:]  # Highest errors in normals
        print(f"🟡 TOP-{top_k} FALSE POSITIVES (Normal → Pathology):")
        for i, idx in enumerate(fp_indices):
            print(f"   [{i+1}] {normal_files[idx]}: error = {normal_errors[idx]:.6f}")
    
    # False Negatives (Pathology classified as Normal)  
    pathology_mask = labels == 1
    pathology_errors = errors[pathology_mask]
    pathology_files = np.array(filenames)[pathology_mask]
    
    if len(pathology_errors) > 0:
        fn_indices = np.argsort(pathology_errors)[:top_k]  # Lowest errors in pathology
        print(f"\n🔴 TOP-{top_k} FALSE NEGATIVES (Pathology → Normal):")
        for i, idx in enumerate(fn_indices):
            print(f"   [{i+1}] {pathology_files[idx]}: error = {pathology_errors[idx]:.6f}")
    
    # True Positives (Correctly detected pathology)
    if len(pathology_errors) > 0:
        tp_indices = np.argsort(pathology_errors)[-top_k:]  # Highest errors in pathology
        print(f"\n🟢 TOP-{top_k} TRUE POSITIVES (Pathology correctly detected):")
        for i, idx in enumerate(tp_indices):
            print(f"   [{i+1}] {pathology_files[idx]}: error = {pathology_errors[idx]:.6f}")

analyze_difficult_cases(test_errors, test_labels, test_filenames)

# Summary статистика по типам ошибок
y_pred_optimal = (test_errors > optimal_threshold).astype(int)

print(f"\n📊 ДЕТАЛЬНЫЙ АНАЛИЗ РЕЗУЛЬТАТОВ:")

# True/False statistics
tp = np.sum((test_labels == 1) & (y_pred_optimal == 1))
tn = np.sum((test_labels == 0) & (y_pred_optimal == 0)) 
fp = np.sum((test_labels == 0) & (y_pred_optimal == 1))
fn = np.sum((test_labels == 1) & (y_pred_optimal == 0))

print(f"   True Positives: {tp}")
print(f"   True Negatives: {tn}")
print(f"   False Positives: {fp}")
print(f"   False Negatives: {fn}")

# Error rate analysis
if len(test_errors) > 0:
    normal_error_rate = np.mean(test_errors[test_labels == 0]) if np.sum(test_labels == 0) > 0 else 0
    pathology_error_rate = np.mean(test_errors[test_labels == 1]) if np.sum(test_labels == 1) > 0 else 0
    
    print(f"\n📈 СРЕДНИЕ RECONSTRUCTION ERRORS:")
    print(f"   Normal samples: {normal_error_rate:.6f}")
    print(f"   Pathology samples: {pathology_error_rate:.6f}")
    print(f"   Separation ratio: {pathology_error_rate/normal_error_rate:.2f}x" if normal_error_rate > 0 else "   Separation ratio: N/A")

print(f"\n✅ Error analysis завершен")

# ЭТАП 7: RESULTS SAVING & EXPORT

In [ ]:
# ===================================
# ЯЧЕЙКА 7.1: FINAL MODEL ARTIFACTS SAVING
# ===================================

print("💾 СОХРАНЕНИЕ ФИНАЛЬНЫХ АРТЕФАКТОВ МОДЕЛИ")
print("=" * 60)

import datetime

# Создаем финальную папку результатов
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
final_results_dir = f"medvae_baseline_results_{timestamp}"
os.makedirs(final_results_dir, exist_ok=True)

print(f"📁 Results directory: {final_results_dir}")

# 1. Сохраняем финальную модель
print("💾 Сохраняем финальную модель...")
final_model_path = f"{final_results_dir}/final_medvae_model.pth"

torch.save({
    'model_state_dict': medvae_model.state_dict(),
    'model_config': {
        'model_type': 'MedVAE3D' if 'MedVAE3D' in str(type(medvae_model)) else 'MockMedVAE3D',
        'input_shape': (1, 128, 128, 64),
        'compression_factor': 64
    },
    'training_config': TRAINING_CONFIG.copy(),
    'final_metrics': {
        'test_auc': float(auc_score) if 'auc_score' in locals() else 0.0,
        'best_f1': float(best_f1) if 'best_f1' in locals() else 0.0,
        'optimal_threshold': float(optimal_threshold) if 'optimal_threshold' in locals() else 0.0
    },
    'timestamp': timestamp
}, final_model_path)

print(f"✅ Model saved: {final_model_path}")

# 2. Сохраняем детальные результаты
print("📊 Сохраняем детальные результаты...")
detailed_results = {
    'experiment_info': {
        'timestamp': timestamp,
        'model_type': 'MedVAE3D' if 'MedVAE3D' in str(type(medvae_model)) else 'MockMedVAE3D',
        'dataset_info': {
            'train_size': len(train_dataset),
            'val_size': len(val_dataset), 
            'test_size': len(test_dataset),
            'total_files': len(train_dataset) + len(val_dataset) + len(test_dataset)
        }
    },
    
    'training_results': {
        'num_epochs_trained': len(training_metrics['train_loss']),
        'best_epoch': training_metrics.get('best_epoch', 0),
        'best_val_loss': training_metrics.get('best_val_loss', float('inf')),
        'final_train_loss': training_metrics['train_loss'][-1] if training_metrics['train_loss'] else 0,
        'final_val_loss': training_metrics['val_loss'][-1] if training_metrics['val_loss'] else 0,
        'total_training_time_minutes': training_metrics.get('total_training_time', 0) / 60
    },
    
    'test_results': {
        'test_auc': float(auc_score) if 'auc_score' in locals() else 0.0,
        'best_f1_score': float(best_f1) if 'best_f1' in locals() else 0.0,
        'optimal_threshold': float(optimal_threshold) if 'optimal_threshold' in locals() else 0.0,
        'confusion_matrix': {
            'true_positives': int(tp) if 'tp' in locals() else 0,
            'true_negatives': int(tn) if 'tn' in locals() else 0,
            'false_positives': int(fp) if 'fp' in locals() else 0,
            'false_negatives': int(fn) if 'fn' in locals() else 0
        },
        'additional_metrics': {
            'sensitivity': float(sensitivity) if 'sensitivity' in locals() else 0.0,
            'specificity': float(specificity) if 'specificity' in locals() else 0.0,
            'ppv': float(ppv) if 'ppv' in locals() else 0.0,
            'npv': float(npv) if 'npv' in locals() else 0.0
        }
    },
    
    'per_sample_results': {
        'filenames': test_filenames,
        'reconstruction_errors': test_errors.tolist(),
        'ground_truth_labels': test_labels.tolist(),
        'predicted_labels': y_pred_optimal.tolist() if 'y_pred_optimal' in locals() else []
    }
}

# Убираем model из training_config для JSON
if 'model' in detailed_results['training_results']:
    del detailed_results['training_results']['model']

results_json_path = f"{final_results_dir}/detailed_results.json"
with open(results_json_path, 'w') as f:
    json.dump(detailed_results, f, indent=2)

print(f"✅ Detailed results saved: {results_json_path}")

# 3. Копируем лучшие визуализации
print("🎨 Копируем визуализации...")
import shutil

# Training progress
if os.path.exists(f"{TRAINING_CONFIG['checkpoint_dir']}/training_progress.png"):
    shutil.copy(f"{TRAINING_CONFIG['checkpoint_dir']}/training_progress.png", 
                f"{final_results_dir}/training_progress.png")

# Final analysis
if os.path.exists(f"{TRAINING_CONFIG['checkpoint_dir']}/final_analysis.png"):
    shutil.copy(f"{TRAINING_CONFIG['checkpoint_dir']}/final_analysis.png",
                f"{final_results_dir}/final_analysis.png")

print("✅ Visualizations copied")

# 4. Создаем Excel report для практического использования
print("📊 Создаем Excel report...")

try:
    import pandas as pd
    
    # Results summary
    summary_data = {
        'Metric': ['ROC AUC', 'Best F1-Score', 'Sensitivity', 'Specificity', 'PPV', 'NPV', 'Optimal Threshold'],
        'Value': [
            auc_score if 'auc_score' in locals() else 0.0,
            best_f1 if 'best_f1' in locals() else 0.0,
            sensitivity if 'sensitivity' in locals() else 0.0,
            specificity if 'specificity' in locals() else 0.0,
            ppv if 'ppv' in locals() else 0.0,
            npv if 'npv' in locals() else 0.0,
            optimal_threshold if 'optimal_threshold' in locals() else 0.0
        ]
    }
    
    summary_df = pd.DataFrame(summary_data)
    
    # Per-sample results
    sample_results_df = pd.DataFrame({
        'Filename': test_filenames,
        'Reconstruction_Error': test_errors,
        'Ground_Truth': ['Normal' if label == 0 else 'Pathology' for label in test_labels],
        'Predicted': ['Normal' if pred == 0 else 'Pathology' for pred in y_pred_optimal] if 'y_pred_optimal' in locals() else ['Unknown'] * len(test_filenames),
        'Correct_Prediction': [gt == pred for gt, pred in zip(test_labels, y_pred_optimal)] if 'y_pred_optimal' in locals() else [False] * len(test_filenames)
    })
    
    # Сохраняем в Excel
    excel_path = f"{final_results_dir}/medvae_baseline_report.xlsx"
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        summary_df.to_excel(writer, sheet_name='Summary_Metrics', index=False)
        sample_results_df.to_excel(writer, sheet_name='Per_Sample_Results', index=False)
    
    print(f"✅ Excel report saved: {excel_path}")
    
except ImportError:
    print("⚠️ pandas не установлен, Excel report пропущен")

print(f"\n✅ Все артефакты сохранены в: {final_results_dir}")


In [ ]:
# ===================================
# ЯЧЕЙКА 7.2: MODEL DEPLOYMENT PREPARATION
# ===================================

print("🚀 ПОДГОТОВКА К DEPLOYMENT")
print("=" * 60)

# Создаем inference класс для production использования
class MedVAEAnomalyDetector:
    """Production-ready класс для anomaly detection"""
    
    def __init__(self, model_path, device='cuda'):
        self.device = torch.device(device if torch.cuda.is_available() else 'cpu')
        self.model = None
        self.config = None
        self.threshold = None
        
        self.load_model(model_path)
    
    def load_model(self, model_path):
        """Загружает модель и конфигурацию"""
        checkpoint = torch.load(model_path, map_location=self.device)
        
        # Определяем тип модели и создаем instance
        model_type = checkpoint.get('model_config', {}).get('model_type', 'MockMedVAE3D')
        
        if model_type == 'MedVAE3D':
            try:
                from medvae import MedVAE3D
                self.model = MedVAE3D.from_pretrained("compression_64")
            except ImportError:
                # Fallback на Mock
                print("Не получилось загрузить")
        
        # Загружаем веса
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.model.to(self.device)
        self.model.eval()
        
        # Сохраняем конфигурацию
        self.config = checkpoint.get('model_config', {})
        self.threshold = checkpoint.get('final_metrics', {}).get('optimal_threshold', 0.001)
        
        print(f"✅ Model loaded: {model_type}")
        print(f"🎯 Optimal threshold: {self.threshold:.6f}")
    
    def predict_single(self, ct_volume):
        """Prediction для одного КТ тома
        
        Args:
            ct_volume: numpy array (H, W, D) или torch tensor (1, H, W, D)
        
        Returns:
            dict: prediction результаты
        """
        # Preprocessing
        if isinstance(ct_volume, np.ndarray):
            ct_tensor = torch.from_numpy(ct_volume.astype(np.float32))
        else:
            ct_tensor = ct_volume
        
        # Добавляем dimensions если нужно
        if len(ct_tensor.shape) == 3:  # (H, W, D)
            ct_tensor = ct_tensor.unsqueeze(0).unsqueeze(0)  # (1, 1, H, W, D)
        elif len(ct_tensor.shape) == 4:  # (1, H, W, D)
            ct_tensor = ct_tensor.unsqueeze(0)  # (1, 1, H, W, D)
        
        ct_tensor = ct_tensor.to(self.device)
        
        # Inference
        with torch.no_grad():
            reconstructed = self.model(ct_tensor)
            mse_error = torch.mean((ct_tensor - reconstructed) ** 2).item()
        
        # Prediction
        is_anomaly = mse_error > self.threshold
        confidence = abs(mse_error - self.threshold) / self.threshold
        
        return {
            'reconstruction_error': mse_error,
            'is_anomaly': is_anomaly,
            'prediction': 'Pathology' if is_anomaly else 'Normal',
            'confidence': confidence,
            'threshold': self.threshold
        }
    
    def predict_batch(self, ct_volumes):
        """Batch prediction"""
        results = []
        for volume in ct_volumes:
            results.append(self.predict_single(volume))
        return results

# Создаем deployment instance
deployment_model_path = f"{final_results_dir}/final_medvae_model.pth"
detector = MedVAEAnomalyDetector(deployment_model_path, device=device)

print(f"✅ Deployment detector создан")

# Тестируем на нескольких примерах
print(f"\n🧪 ТЕСТИРОВАНИЕ DEPLOYMENT MODEL:")

test_samples = 3
for i in range(min(test_samples, len(test_dataset))):
    sample = test_dataset[i]
    ct_volume = sample['image']  # (1, 128, 128, 64)
    true_label = sample['label']
    filename = sample['filename']
    
    # Prediction
    result = detector.predict_single(ct_volume)
    
    print(f"\n📋 Sample {i+1}: {filename}")
    print(f"   True label: {'Normal' if true_label == 0 else 'Pathology'}")
    print(f"   Predicted: {result['prediction']}")
    print(f"   Reconstruction error: {result['reconstruction_error']:.6f}")
    print(f"   Confidence: {result['confidence']:.3f}")
    print(f"   Correct: {result['prediction'] == ('Normal' if true_label == 0 else 'Pathology')}")

print(f"\n✅ Deployment testing завершен")

In [ ]:
# ===================================
# ЯЧЕЙКА 7.3: FINAL SUMMARY И CLEANUP
# ===================================

print("🎉 ФИНАЛЬНОЕ РЕЗЮМЕ И CLEANUP")
print("=" * 60)

# Финальная статистика
print("📊 ИТОГОВЫЕ РЕЗУЛЬТАТЫ ЭКСПЕРИМЕНТА:")
print(f"   🕒 Timestamp: {timestamp}")
print(f"   📁 Results directory: {final_results_dir}")

if 'auc_score' in locals():
    print(f"\n🎯 КЛЮЧЕВЫЕ МЕТРИКИ:")
    print(f"   ROC AUC: {auc_score:.4f}")
    print(f"   Best F1-Score: {best_f1:.4f}")
    print(f"   Optimal Threshold: {optimal_threshold:.6f}")
    
    print(f"\n📊 CONFUSION MATRIX:")
    print(f"   True Positives: {tp}")
    print(f"   True Negatives: {tn}")
    print(f"   False Positives: {fp}")
    print(f"   False Negatives: {fn}")
    
    print(f"\n📈 CLINICAL METRICS:")
    print(f"   Sensitivity: {sensitivity:.4f}")
    print(f"   Specificity: {specificity:.4f}")
    print(f"   PPV: {ppv:.4f}")
    print(f"   NPV: {npv:.4f}")

# Training summary
if training_metrics['train_loss']:
    print(f"\n🏋️ TRAINING SUMMARY:")
    print(f"   Epochs trained: {len(training_metrics['train_loss'])}")
    print(f"   Best epoch: {training_metrics.get('best_epoch', 'N/A')}")
    print(f"   Total training time: {training_metrics.get('total_training_time', 0)/60:.1f} minutes")
    print(f"   Final train loss: {training_metrics['train_loss'][-1]:.6f}")
    print(f"   Final val loss: {training_metrics['val_loss'][-1]:.6f}")

# Dataset summary
print(f"\n📁 DATASET SUMMARY:")
print(f"   Training: {len(train_dataset)} files (100% normal)")
print(f"   Validation: {len(val_dataset)} files")
print(f"   Test: {len(test_dataset)} files")
print(f"   Total processed: {len(train_dataset) + len(val_dataset) + len(test_dataset)} files")

# Files generated
print(f"\n📄 GENERATED FILES:")
files_in_results = os.listdir(final_results_dir) if os.path.exists(final_results_dir) else []
for file in files_in_results:
    file_path = os.path.join(final_results_dir, file)
    size_mb = os.path.getsize(file_path) / (1024*1024)
    print(f"   📄 {file} ({size_mb:.1f} MB)")

# Next steps recommendation
print(f"\n🚀 РЕКОМЕНДАЦИИ ДЛЯ ДАЛЬНЕЙШЕГО РАЗВИТИЯ:")
print(f"   1. Эксперименты с различными архитектурами (Swin UNETR)")
print(f"   2. Ensemble подходы для улучшения performance")
print(f"   3. Локализация патологий через attention maps")
print(f"   4. Integration в clinical workflow")
print(f"   5. Валидация на дополнительных datasets")

# Cleanup и memory management
print(f"\n🧹 CLEANUP:")

# Освобождаем GPU memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    memory_allocated = torch.cuda.memory_allocated(device) / 1024**2
    print(f"   GPU memory after cleanup: {memory_allocated:.1f} MB")

# Удаляем промежуточные переменные
cleanup_vars = ['test_errors', 'test_labels', 'val_errors', 'val_labels', 
                'normal_errors', 'pathology_errors', 'training_metrics']

for var_name in cleanup_vars:
    if var_name in locals():
        del locals()[var_name]

print(f"   ✅ Cleanup завершен")

# Success message
print(f"\n" + "="*60)
print(f"🎊 MEDVAE BASELINE УСПЕШНО ЗАВЕРШЕН")
print(f"="*60)

if 'auc_score' in locals() and auc_score >= 0.85:
    print(f"🏆 ОТЛИЧНЫЙ РЕЗУЛЬТАТ: AUC {auc_score:.4f} >= 0.85")
elif 'auc_score' in locals() and auc_score >= 0.80:
    print(f"✅ ХОРОШИЙ РЕЗУЛЬТАТ: AUC {auc_score:.4f} >= 0.80")
else:
    print(f"📊 BASELINE УСТАНОВЛЕН: AUC {auc_score:.4f if 'auc_score' in locals() else 'N/A'}")

print(f"\n📁 Все результаты сохранены в: {final_results_dir}")
print(f"🚀 Готово к следующему этапу: Swin UNETR enhancement!")
print(f"="*60)